In [633]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)
import datetime as dt
import sqlite3
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from scipy import stats


In [634]:
# Connect to the database
conn = sqlite3.connect('nba_data.db')

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

In [635]:
# List all tables in the database
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
tables

[('passing_df_data',),
 ('rebounding_df_data',),
 ('drives_df_data',),
 ('catchshoot_df_data',),
 ('pullup_df_data',),
 ('speeddistance_data',),
 ('posttouch_data',),
 ('player_type_defensive',),
 ('catchshoot_player_data',),
 ('pullup_player_data',),
 ('passing_data',),
 ('rebounding_player_data',),
 ('drives_player_data',),
 ('speeddistance_player_data',),
 ('posttouch_player_data',),
 ('painttouch_player_data',),
 ('shotclock_data',),
 ('closestdefender_data',),
 ('dribbles_shot_data',),
 ('touchtime_shot_data',),
 ('gamelogs',),
 ('player_points_scores',),
 ('player_3s_cluster',),
 ('player_3s_scores',),
 ('opponent_def_pts_rankings',),
 ('opponent_def_3pt_rankings',),
 ('opponent_def_type_scores',),
 ('closest_def_total',),
 ('playtype_off_reformat',),
 ('shotdetail_player_rolling',),
 ('drives_player_rolling',),
 ('catchshoot_player_rolling',),
 ('pullup_player_rolling',),
 ('rebounding_player_rolling',),
 ('speeddistance_player_rolling',),
 ('drives_team_def_rolling',),
 ('catch

In [636]:
table_name = "shotclock_data"

In [637]:
shotclock_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [638]:
today = pd.Timestamp.today()


In [639]:
from nba_api.stats.endpoints import leaguegamefinder


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2023-24',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_23 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_23.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_23['GAME_DATE'] = pd.to_datetime(game_sched_23['GAME_DATE'])

In [640]:
from nba_api.stats.endpoints import leaguegamefinder


# Fetch all games for the 2023-24 season
gamefinder = leaguegamefinder.LeagueGameFinder(season_nullable='2024-25',league_id_nullable='00',season_type_nullable='Regular Season')
games = gamefinder.get_data_frames()[0]

# Display the first few rows
games_1 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
games_2 = games[['GAME_ID', 'GAME_DATE', 'SEASON_ID', 'TEAM_ID', 'TEAM_ABBREVIATION', 'TEAM_NAME']]
game_sched1 = games_1.merge(games_2, how='left',on=['GAME_ID','GAME_DATE','SEASON_ID'])
game_sched_24 = pd.DataFrame(game_sched1.loc[game_sched1['TEAM_ID_x']!=game_sched1['TEAM_ID_y']]).reset_index(drop=True)
game_sched_24.rename(columns={'TEAM_ID_x':'TEAM_ID','TEAM_ABBREVIATION_x':'TEAM_ABBREVIATION','TEAM_NAME_x':'TEAM_NAME','TEAM_ID_y':'OPPONENT_ID','TEAM_ABBREVIATION_y':'OPPONENT_ABBREVIATION','TEAM_NAME_y':'OPPONENT_NAME'},inplace=True)
game_sched_24['GAME_DATE'] = pd.to_datetime(game_sched_24['GAME_DATE'])

In [641]:
game_sched = pd.concat([game_sched_23, game_sched_24])

In [642]:
#game_sched['GAME_DATE'] = pd.to_datetime(game_sched['GAME_DATE'])

In [643]:
table_name = "player_scoring_clusters_new"
pts_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_3s_cluster_new"
threes_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_points_scores_new"
pts_scores_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_3s_scores"
threes_scores_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_passing_clusters"
pass_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

table_name = "player_rebounding_clusters"
reb_cluster_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [644]:
pts_cluster_data['as_of'] = pd.to_datetime(pts_cluster_data['as_of'])
threes_cluster_data['as_of'] = pd.to_datetime(threes_cluster_data['as_of'])
pts_scores_data['as_of'] = pd.to_datetime(pts_scores_data['as_of'])
threes_scores_data['as_of'] = pd.to_datetime(threes_scores_data['as_of'])
pass_cluster_data['as_of'] = pd.to_datetime(pass_cluster_data['as_of'])
reb_cluster_data['as_of'] = pd.to_datetime(reb_cluster_data['as_of'])

max_date_pts_c = pts_cluster_data['as_of'].max()
max_date_3s_c = threes_cluster_data['as_of'].max()
max_date_pts_s = pts_scores_data['as_of'].max()
max_date_3s_s = threes_scores_data['as_of'].max()
max_date_asts_s = pass_cluster_data['as_of'].max()
max_date_reb_s = reb_cluster_data['as_of'].max()

pts_cluster_df = pts_cluster_data[pts_cluster_data['as_of'] == max_date_pts_c]
threes_cluster_df = threes_cluster_data[threes_cluster_data['as_of'] == max_date_3s_c]
pts_scores_df = pts_scores_data[pts_scores_data['as_of'] == max_date_pts_s]
threes_scores_df = threes_scores_data[threes_scores_data['as_of'] == max_date_3s_s]
ast_cluster_df = pass_cluster_data[pass_cluster_data['as_of'] == max_date_asts_s]
reb_cluster_df = reb_cluster_data[reb_cluster_data['as_of'] == max_date_reb_s]

In [645]:
#ast_cluster_df.loc[ast_cluster_df["PLAYER_ID"]==202710]

In [646]:
pts_cluster_add = pts_cluster_df[['PLAYER_ID','Cluster']].rename(columns={'Cluster':'Cluster_Pts'})
threes_cluster_add = threes_cluster_df[['PLAYER_ID','Cluster']].rename(columns={'Cluster':'Cluster_3pt'})

In [647]:
table_name = "gamelogs"
gamelogs_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [648]:
gamelogs_data['GAME_DATE'] = pd.to_datetime(gamelogs_data['GAME_DATE'])

In [649]:
#ast_cluster_df[['PLAYER_ID','Passing_Cluster']]

In [650]:
gamelogs_data = gamelogs_data.merge(pts_cluster_add, how='left')
gamelogs_data = gamelogs_data.merge(threes_cluster_add, how='left')
gamelogs_data = gamelogs_data.merge(ast_cluster_df[['PLAYER_ID','Passing_Cluster']], how='left')
gamelogs_data = gamelogs_data.merge(reb_cluster_df[['PLAYER_ID','Rebounding_Cluster']], how='left')

In [651]:
gamelogs_data

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,id,Cluster_Pts,Cluster_3pt,Passing_Cluster,Rebounding_Cluster
0,2023-24,1630173,Precious Achiuwa,Precious,1610612752,NYK,New York Knicks,0022301190,2024-04-14,NYK vs. CHI,W,18.506667,2,3,0.667,0,0,0.00,0,0,0.0,1,4,5,2,2,0,1,0,3,0,4,-2,14.0,0,0,13.0,1,1,1,1,44,40,55,16,23,52,23,34,39,34,47,26,41,16,11,32,19,1,51,45,49,37,55,10,1,53,1,18:30,1630173_2024-04-14T00:00:00_1610612752,NaN,NaN,NaN,2.0
1,2023-24,1630173,Precious Achiuwa,Precious,1610612752,NYK,New York Knicks,0022301175,2024-04-12,NYK vs. BKN,W,7.566667,2,2,1.000,1,1,1.00,0,0,0.0,0,3,3,0,0,0,0,0,0,0,5,3,8.6,0,0,9.0,1,1,1,1,69,40,61,1,4,29,1,34,39,34,61,42,58,49,49,32,43,1,1,45,45,26,63,10,1,61,1,7:34,1630173_2024-04-12T00:00:00_1610612752,NaN,NaN,NaN,2.0
2,2023-24,1630173,Precious Achiuwa,Precious,1610612752,NYK,New York Knicks,0022301167,2024-04-11,NYK @ BOS,W,16.033333,1,6,0.167,0,1,0.00,0,0,0.0,2,3,5,0,1,0,1,2,0,0,2,-9,10.0,0,0,9.0,1,1,1,1,55,55,36,64,23,29,23,34,39,34,39,42,41,49,24,32,19,67,1,45,58,55,61,10,1,61,1,16:02,1630173_2024-04-11T00:00:00_1610612752,NaN,NaN,NaN,2.0
3,2023-24,1630173,Precious Achiuwa,Precious,1610612752,NYK,New York Knicks,0022301139,2024-04-07,NYK @ MIL,W,4.950000,0,1,0.000,0,0,0.00,0,0,0.0,0,0,0,0,0,0,0,0,1,0,0,5,0.0,0,0,0.0,1,1,1,1,73,66,70,66,23,52,23,34,39,34,61,72,73,49,49,32,43,1,15,45,67,20,74,10,1,74,1,4:57,1630173_2024-04-07T00:00:00_1610612752,NaN,NaN,NaN,2.0
4,2023-24,1630173,Precious Achiuwa,Precious,1610612752,NYK,New York Knicks,0022301119,2024-04-05,NYK @ CHI,L,18.580000,0,2,0.000,0,1,0.00,0,0,0.0,1,3,4,1,1,1,0,1,4,0,0,-2,8.3,0,0,7.0,1,41,41,41,42,66,61,66,23,29,23,34,39,34,47,42,53,23,24,13,43,46,63,45,67,37,64,10,1,64,1,18:35,1630173_2024-04-05T00:00:00_1610612752,NaN,NaN,NaN,2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47082,2024-25,1641783,Tristan da Silva,Tristan,1610612753,ORL,Orlando Magic,0022401141,2025-03-13,ORL @ NOP,W,16.633333,1,5,0.200,1,4,0.25,0,0,0.0,0,2,2,1,1,0,1,0,1,0,3,-2,8.9,0,0,9.0,1,1,1,1,1,2,1,2,1,1,1,2,2,2,1,2,2,1,1,2,1,1,3,2,2,2,2,1,1,2,1,16:38,1641783_2025-03-13T00:00:00_1610612753,NaN,4.0,NaN,NaN
47083,2024-25,1641783,Tristan da Silva,Tristan,1610612753,ORL,Orlando Magic,0022400934,2025-03-10,ORL @ HOU,L,9.716667,0,3,0.000,0,1,0.00,0,0,0.0,0,1,1,0,0,0,0,0,0,0,0,-5,1.2,0,0,1.0,1,3,3,3,3,3,3,3,2,3,2,2,2,2,1,3,3,3,2,2,2,1,1,2,3,3,3,1,1,3,1,9:43,1641783_2025-03-10T00:00:00_1610612753,NaN,4.0,NaN,NaN
47084,2024-25,1641783,Tristan da Silva,Tristan,1610612753,ORL,Orlando Magic,0022400917,2025-03-08,ORL @ MIL,W,10.983333,2,4,0.500,0,2,0.00,1,1,1.0,0,3,3,1,0,1,0,0,0,1,5,4,13.1,0,0,11.0,1,1,1,1,2,1,2,1,2,2,2,1,1,1,1,1,1,1,2,1,2,1,1,1,1,1,1,1,1,1,1,10:59,1641783_2025-03-08T00:00:00_1610612753,NaN,4.0,NaN,NaN
47085,2024-25,1628427,Vlatko Čančar,Vlatko,1610612743,DEN,Denver Nuggets,0022400952,2025-03-12,DEN vs. MIN,L,5.083333,0,0,0.000,0,0,0.00,0,0,0.0,0,2,2,0,0,0,0,0,0,0,0,6,2.4,0,0,2.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,5:05,1628427_2025-03-12T00:00:00_1610612743,NaN,NaN,NaN,NaN


In [652]:
 gamelogs_sorted = pd.DataFrame(gamelogs_data.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [653]:

def calculate_weighted_rolling_stats(
    gamelogs: pd.DataFrame,
    rolling_columns: list,
    window_size: int = 60,
    method: str = 'average'
) -> pd.DataFrame:
    """
    Calculate rolling statistics for basketball statistics.
    
    Parameters:
    -----------
    gamelogs : pd.DataFrame
        DataFrame containing game logs with required columns:
        - OPPONENT_ID
        - GAME_DATE
        - Statistical columns specified in rolling_columns
    rolling_columns : list
        List of statistical columns to calculate rolling stats for
    window_size : int
        Number of games to include in rolling window
    method : str
        'average' for rolling mean, 'sum' for rolling sum
    """
    if method not in ['average', 'sum']:
        raise ValueError("method must be either 'average' or 'sum'")
    
    # Validate required columns
    required_columns = {'PLAYER_ID', 'GAME_DATE'}
    missing_columns = required_columns - set(gamelogs.columns)
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    
    # Validate statistical columns
    missing_stat_columns = set(rolling_columns) - set(gamelogs.columns)
    if missing_stat_columns:
        raise ValueError(f"DataFrame is missing specified statistical columns: {missing_stat_columns}")
    
    # Ensure gamelogs are sorted by date
    gamelogs['GAME_DATE'] = pd.to_datetime(gamelogs['GAME_DATE'])
    gamelogs_sorted = gamelogs.sort_values(by=['PLAYER_ID', 'GAME_DATE']).reset_index(drop=True)
    
    # Initialize list to store rolling calculations
    rolling_dfs = []
    
    # Calculate games count for each window
    games_in_window = (
        gamelogs_sorted
        .groupby('PLAYER_ID')['GAME_DATE']
        .rolling(window=window_size, min_periods=1)
        .count()
        .reset_index(level=0, drop=True)
        .rename(f'GAMES_IN_WINDOW_{window_size}G')
    )
    rolling_dfs.append(games_in_window)
    
    # Calculate rolling statistics for each statistical column
    for col in rolling_columns:
        try:
            if method == 'average':
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .mean()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Mavg'
            else:  # method == 'sum'
                rolling_col = (
                    gamelogs_sorted
                    .groupby('PLAYER_ID')[col]
                    .rolling(window=window_size, min_periods=1)
                    .sum()
                    .reset_index(level=0, drop=True)
                )
                suffix = f'{window_size}G_Sum'
            
            rolling_dfs.append(rolling_col.rename(f'{col}_{suffix}'))
            
        except Exception as e:
            print(f"Error processing column '{col}': {str(e)}")
            continue
    
    if len(rolling_dfs) <= 1:  # Only games count column
        raise ValueError("No statistical columns were successfully processed")
    
    # Combine all rolling statistics with original data
    rolling_df = pd.concat(rolling_dfs, axis=1)
    result_df = pd.concat(
        [gamelogs_sorted.reset_index(drop=True), rolling_df],
        axis=1
    )
    
    return result_df

In [654]:
def calculate_team_rolling_stats(
    gamelogs: pd.DataFrame,
    rolling_columns: list,
    group_columns: list = ['OPPONENT_ID'],
    window_size: int = 60,
    method: str = 'average'
) -> pd.DataFrame:
    """
    Calculate rolling statistics for basketball statistics with multiple grouping columns.
    
    Parameters:
    -----------
    gamelogs : pd.DataFrame
        DataFrame containing game logs with required columns:
        - Grouping columns specified in group_columns
        - GAME_DATE
        - Statistical columns specified in rolling_columns
    rolling_columns : list
        List of statistical columns to calculate rolling stats for
    group_columns : list
        List of columns to group by before calculating rolling statistics
    window_size : int
        Number of games to include in rolling window
    method : str
        'average' for rolling mean, 'sum' for rolling sum
    """
    if method not in ['average', 'sum']:
        raise ValueError("method must be either 'average' or 'sum'")
    
    # Validate required columns
    required_columns = set(["OPPONENT_ID"]) | {'GAME_DATE'}
    missing_columns = required_columns - set(gamelogs.columns)
    if missing_columns:
        raise ValueError(f"DataFrame is missing required columns: {missing_columns}")
    
    # Validate statistical columns
    missing_stat_columns = set(rolling_columns) - set(gamelogs.columns)
    if missing_stat_columns:
        raise ValueError(f"DataFrame is missing specified statistical columns: {missing_stat_columns}")
    
    # Ensure gamelogs are sorted by date within groups
    gamelogs['GAME_DATE'] = pd.to_datetime(gamelogs['GAME_DATE'])
    sort_columns = group_columns + ['GAME_DATE']
    gamelogs_sorted = gamelogs.sort_values(by=sort_columns).reset_index(drop=True)
    
    # Initialize list to store rolling calculations
    rolling_dfs = []
    
    # Calculate games count for each window using all grouping columns
    games_in_window = (
        gamelogs_sorted
        .groupby(group_columns)['GAME_DATE']
        .rolling(window=window_size, min_periods=1)
        .count()
        .reset_index(level=list(range(len(group_columns))), drop=True)
        .rename(f'GAMES_IN_WINDOW_TEAM_{window_size}G')
    )
    rolling_dfs.append(games_in_window)
    
    # Calculate rolling statistics for each statistical column
    for col in rolling_columns:
        try:
            if method == 'average':
                rolling_col = (
                    gamelogs_sorted
                    .groupby(group_columns)[col]
                    .rolling(window=window_size, min_periods=1)
                    .mean()
                    .reset_index(level=list(range(len(group_columns))), drop=True)
                )
                suffix = f'{window_size}G_Mavg'
            else:  # method == 'sum'
                rolling_col = (
                    gamelogs_sorted
                    .groupby(group_columns)[col]
                    .rolling(window=window_size, min_periods=1)
                    .sum()
                    .reset_index(level=list(range(len(group_columns))), drop=True)
                )
                suffix = f'{window_size}G_Sum'
            
            rolling_dfs.append(rolling_col.rename(f'{col}_{suffix}'))
            
        except Exception as e:
            print(f"Error processing column '{col}': {str(e)}")
            continue
    
    if len(rolling_dfs) <= 1:  # Only games count column
        raise ValueError("No statistical columns were successfully processed")
    
    # Combine all rolling statistics with original data
    rolling_df = pd.concat(rolling_dfs, axis=1)
    result_df = pd.concat(
        [gamelogs_sorted.reset_index(drop=True), rolling_df],
        axis=1
    )
    
    return result_df

In [655]:
gamelogs_with_rolling = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted,
    rolling_columns=['MIN', 'PTS', 'REB', 'AST', 'FGA','FG3M','FG3A'])

In [656]:
gamelogs_with_rolling

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,FGA,FG_PCT,FG3M,FG3A,FG3_PCT,FTM,FTA,FT_PCT,OREB,DREB,REB,AST,TOV,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,NBA_FANTASY_PTS,DD2,TD3,WNBA_FANTASY_PTS,GP_RANK,W_RANK,L_RANK,W_PCT_RANK,MIN_RANK,FGM_RANK,FGA_RANK,FG_PCT_RANK,FG3M_RANK,FG3A_RANK,FG3_PCT_RANK,FTM_RANK,FTA_RANK,FT_PCT_RANK,OREB_RANK,DREB_RANK,REB_RANK,AST_RANK,TOV_RANK,STL_RANK,BLK_RANK,BLKA_RANK,PF_RANK,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,id,Cluster_Pts,Cluster_3pt,Passing_Cluster,Rebounding_Cluster,GAMES_IN_WINDOW_60G,MIN_60G_Mavg,PTS_60G_Mavg,REB_60G_Mavg,AST_60G_Mavg,FGA_60G_Mavg,FG3M_60G_Mavg,FG3A_60G_Mavg
0,2023-24,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,0022300061,2023-10-24,LAL @ DEN,L,29.010000,10,16,0.625,1,4,0.250,0,1,0.000,1,7,8,5,0,1,0,1,1,1,21,7,41.1,0,0,37.0,1,42,42,42,62,26,46,14,47,43,54,68,67,68,17,20,22,60,70,23,31,28,22,65,53,29,53,28,6,57,1,29:01,2544_2023-10-24T00:00:00_1610612747,5.0,2.0,1.0,3.0,1.0,29.010000,21.000000,8.000000,5.000000,16.000000,1.000000,4.000000
1,2023-24,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,0022300076,2023-10-26,LAL vs. PHX,W,35.000000,7,14,0.500,1,5,0.200,6,8,0.750,1,7,8,9,5,2,2,0,1,4,21,22,51.1,0,0,47.0,1,1,1,1,50,54,56,44,47,27,59,17,14,37,17,20,22,19,5,12,2,1,22,39,53,5,26,28,6,29,1,35:00,2544_2023-10-26T00:00:00_1610612747,5.0,2.0,1.0,3.0,2.0,32.005000,21.000000,8.000000,7.000000,15.000000,1.000000,4.500000
2,2023-24,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,0022300100,2023-10-29,LAL @ SAC,L,39.083333,11,19,0.579,3,8,0.375,2,6,0.333,0,15,15,8,8,0,0,0,2,3,27,-5,49.0,1,0,53.0,1,42,42,42,10,21,24,27,10,7,41,52,29,66,41,2,2,34,1,50,31,1,53,49,26,47,34,1,6,18,1,39:05,2544_2023-10-29T00:00:00_1610612747,5.0,2.0,1.0,3.0,3.0,34.364444,23.000000,10.333333,7.333333,16.333333,1.666667,5.666667
3,2023-24,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,0022300111,2023-10-30,LAL vs. ORL,W,32.783333,7,17,0.412,2,6,0.333,3,4,0.750,0,3,3,4,5,3,1,0,0,2,19,5,35.6,0,0,36.0,1,1,1,1,57,54,39,62,24,18,44,46,47,37,41,66,68,68,5,5,8,1,1,58,59,34,61,28,6,61,1,32:47,2544_2023-10-30T00:00:00_1610612747,5.0,2.0,1.0,3.0,4.0,33.969167,22.000000,8.500000,6.500000,16.500000,1.750000,5.750000
4,2023-24,2544,LeBron James,LeBron,1610612747,LAL,Los Angeles Lakers,0022300127,2023-11-01,LAL vs. LAC,W,42.483333,13,19,0.684,4,8,0.500,5,10,0.500,0,12,12,7,4,1,2,0,3,6,35,6,64.9,1,0,64.0,1,1,1,1,2,6,24,6,5,7,14,22,4,59,41,4,5,45,16,23,2,1,65,8,7,32,7,1,6,4,1,42:29,2544_2023-11-01T00:00:00_1610612747,5.0,2.0,1.0,3.0,5.0,35.672000,24.600000,9.200000,6.600000,17.000000,2.200000,6.200000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47082,2024-25,1642530,Yuki Kawamura,Yuki,1610612763,MEM,Memphis Grizzlies,0022400440,2024-12-29,MEM @ OKC,L,11.016667,4,5,0.800,2,3,0.667,0,0,0.000,0,3,3,3,0,0,0,0,1,1,10,5,18.1,0,0,18.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,11:01,1642530_2024-12-29T00:00:00_1610612763,NaN,NaN,NaN,NaN,15.0,3.468778,1.600000,0.333333,0.733333,1.333333,0.266667,0.933333
47083,2024-25,1642530,Yuki Kawamura,Yuki,1610612763,MEM,Memphis Grizzlies,0022400499,2025-01-06,MEM vs. DAL,W,2.300000,0,1,0.000,0,1,0.000,0,0,0.000,0,1,1,1,0,0,0,0,2,0,0,1,2.7,0,0,2.0,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,2:18,1642530_2025-01-06T00:00:00_1610612763,NaN,NaN,NaN,NaN,16.0,3.395729,1.500000,0.375000,0.750000,1.312500,0.250000,0.937500
47084,2024-25,1642530,Yuki Kawamura,Yuki,1610612763,MEM,Memphis Grizzlies,0022400582,2025-01-17,MEM @ SAS,W,2.116667,0,0,0.000,0,0,0.000,0,0,0.000,0,0,0,0,0,0,0,0

In [657]:
gamelogs_with_sum = calculate_weighted_rolling_stats(
    gamelogs=gamelogs_sorted,  # Pass in the previous result
    rolling_columns=['MIN', 'PTS'],
    method='sum')

In [658]:
#gamelogs_with_sum

In [659]:
gamelogs_sum_add = pd.DataFrame(gamelogs_with_sum[['PLAYER_ID','GAME_DATE','MIN_60G_Sum','PTS_60G_Sum']])

In [660]:
gamelogs_rolling= gamelogs_with_rolling.merge(gamelogs_sum_add, how='left')

In [661]:
gamelogs_with_rolling_w_opp = gamelogs_rolling.merge(game_sched, how='left')

In [662]:
gamelogs_with_rolling_w_opp.columns

Index(['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME', 'NICKNAME', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
       'WL', 'MIN', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM',
       'FTA', 'FT_PCT', 'OREB', 'DREB', 'REB', 'AST', 'TOV', 'STL', 'BLK',
       'BLKA', 'PF', 'PFD', 'PTS', 'PLUS_MINUS', 'NBA_FANTASY_PTS', 'DD2',
       'TD3', 'WNBA_FANTASY_PTS', 'GP_RANK', 'W_RANK', 'L_RANK', 'W_PCT_RANK',
       'MIN_RANK', 'FGM_RANK', 'FGA_RANK', 'FG_PCT_RANK', 'FG3M_RANK',
       'FG3A_RANK', 'FG3_PCT_RANK', 'FTM_RANK', 'FTA_RANK', 'FT_PCT_RANK',
       'OREB_RANK', 'DREB_RANK', 'REB_RANK', 'AST_RANK', 'TOV_RANK',
       'STL_RANK', 'BLK_RANK', 'BLKA_RANK', 'PF_RANK', 'PFD_RANK', 'PTS_RANK',
       'PLUS_MINUS_RANK', 'NBA_FANTASY_PTS_RANK', 'DD2_RANK', 'TD3_RANK',
       'WNBA_FANTASY_PTS_RANK', 'AVAILABLE_FLAG', 'MIN_SEC', 'id',
       'Cluster_Pts', 'Cluster_3pt', 'Passing_Cluster', 'Rebounding_Cluster',
       'GAMES_IN_WINDOW_60G', '

In [663]:
rolling_opp_gamelogs = pd.DataFrame(gamelogs_with_rolling_w_opp[['SEASON_YEAR', 'PLAYER_ID', 'PLAYER_NAME',  'TEAM_ID',
       'TEAM_ABBREVIATION', 'TEAM_NAME', 'GAME_ID', 'GAME_DATE', 'MATCHUP',
        'MIN', 'FGM', 'FGA', 'FG3M', 'FG3A', 'REB', 'AST', 'TOV', 'STL', 'BLK',
        'PTS', 'Cluster_Pts', 'Cluster_3pt','Passing_Cluster','Rebounding_Cluster','GAMES_IN_WINDOW_60G', 'MIN_60G_Mavg',
       'PTS_60G_Mavg', 'REB_60G_Mavg', 'AST_60G_Mavg', 'FGA_60G_Mavg',
       'FG3M_60G_Mavg', 'FG3A_60G_Mavg', 'MIN_60G_Sum', 'PTS_60G_Sum', 'SEASON_ID', 'OPPONENT_ID', 'OPPONENT_ABBREVIATION',
       'OPPONENT_NAME']])

In [664]:
rolling_opp_cluster_pts = pd.DataFrame(rolling_opp_gamelogs.loc[(rolling_opp_gamelogs['Cluster_Pts'].notna())])

In [665]:
rolling_opp_cluster_pts['PTS_per_MIN_60Gavg'] = rolling_opp_cluster_pts['PTS_60G_Sum'] /rolling_opp_cluster_pts['MIN_60G_Sum'] 

In [666]:
rolling_opp_cluster_pts['PTS_DIFF'] = rolling_opp_cluster_pts['PTS'] - rolling_opp_cluster_pts['PTS_60G_Mavg'].round(2)
rolling_opp_cluster_pts['FG3M_DIFF'] = rolling_opp_cluster_pts['FG3M'] - rolling_opp_cluster_pts['FG3M_60G_Mavg'].round(2)
rolling_opp_cluster_pts['FG3A_DIFF'] = rolling_opp_cluster_pts['FG3A'] - rolling_opp_cluster_pts['FG3A_60G_Mavg'].round(2)
rolling_opp_cluster_pts['AST_DIFF'] = rolling_opp_cluster_pts['AST'] - rolling_opp_cluster_pts['AST_60G_Mavg'].round(2)
rolling_opp_cluster_pts['REB_DIFF'] = rolling_opp_cluster_pts['REB'] - rolling_opp_cluster_pts['REB_60G_Mavg'].round(2)

In [667]:
rolling_opp_cluster_pts['PTS_MIN_DIFF'] = (rolling_opp_cluster_pts['PTS']/rolling_opp_cluster_pts['MIN']) - rolling_opp_cluster_pts['PTS_per_MIN_60Gavg'] 

In [668]:
#rolling_opp_cluster_pts

In [669]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
rolling_opp_cluster_pts['id'] = rolling_opp_cluster_pts['GAME_DATE'].astype(str)+"_"+rolling_opp_cluster_pts['PLAYER_ID'].astype(str)+"_"+rolling_opp_cluster_pts['TEAM_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "player_rolling_gamelog_diff"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    rolling_opp_cluster_pts.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = rolling_opp_cluster_pts[~rolling_opp_cluster_pts['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'player_rolling_gamelog_diff' already exists. Checking for new records...
Inserted 628 new records into 'player_rolling_gamelog_diff'.


In [670]:
opponent_ranking_pts = rolling_opp_cluster_pts.groupby(['OPPONENT_ID', 'OPPONENT_NAME','Cluster_Pts','SEASON_YEAR']).agg(Games=('GAME_DATE', 'count'),
    Avg_PTS_Diff=('PTS_DIFF', 'mean'), Avg_PTS_allowed=('PTS', 'mean') , Avg_PTS_MIN_Diff=('PTS_MIN_DIFF', 'mean')
).reset_index()

In [671]:
opponent_ranking_pts.sort_values(by='Avg_PTS_Diff',ascending=False)

,OPPONENT_ID,OPPONENT_NAME,Cluster_Pts,SEASON_YEAR,Games,Avg_PTS_Diff,Avg_PTS_allowed,Avg_PTS_MIN_Diff
388,1610612764,Washington Wizards,5.0,2023-24,52,3.728654,26.192308,0.097780
248,1610612754,Indiana Pacers,5.0,2023-24,57,3.538772,26.543860,0.093916
359,1610612762,Utah Jazz,4.0,2024-25,49,2.784286,18.571429,0.062273
238,1610612754,Indiana Pacers,0.0,2023-24,85,2.715765,24.858824,0.079070
358,1610612762,Utah Jazz,4.0,2023-24,69,2.710290,17.898551,0.085290
...,...,...,...,...,...,...,...,...
123,1610612745,Houston Rockets,5.0,2024-25,32,-2.773437,20.343750,-0.093686
137,1610612746,LA Clippers,5.0,2024-25,45,-3.032667,19.888889,-0.088317
189,1610612750,Minnesota Timberwolves,3.0,2024-25,36,-3.041389,8.055556,-0.117556
323,1610612760,Oklahoma City Thunder,0.0,2024-25,68,-3.216765,18.279412,-0.096379


In [672]:
opponent_ranking_pts['as_of'] = pd.to_datetime(today)
opponent_ranking_pts['id'] = opponent_ranking_pts['as_of'].astype(str)+"_"+opponent_ranking_pts['OPPONENT_ID'].astype(str)+opponent_ranking_pts['Cluster_Pts'].astype(str)+opponent_ranking_pts['SEASON_YEAR'].astype(str)

In [673]:
opponent_ranking_pts['as_of'] = pd.to_datetime(today)
opponent_ranking_pts['id'] = opponent_ranking_pts['as_of'].astype(str)+"_"+opponent_ranking_pts['OPPONENT_ID'].astype(str)+opponent_ranking_pts['Cluster_Pts'].astype(str)+opponent_ranking_pts['SEASON_YEAR'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "opponent_def_pts_rankings"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    opponent_ranking_pts.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = opponent_ranking_pts[~opponent_ranking_pts['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'opponent_def_pts_rankings' already exists. Checking for new records...
Inserted 420 new records into 'opponent_def_pts_rankings'.


In [674]:
opponent_ranking_3pt = rolling_opp_cluster_pts.groupby(['OPPONENT_ID', 'OPPONENT_NAME','Cluster_3pt','SEASON_YEAR']).agg(Games=('GAME_DATE', 'count'),
    Avg_FG3M_Diff=('FG3M_DIFF', 'mean'), Avg_FG3M_allowed=('FG3M', 'mean') , Avg_FG3A_Diff=('FG3A_DIFF', 'mean')
).reset_index()

In [675]:
opponent_ranking_3pt.sort_values(by='Avg_FG3M_Diff',ascending=False)

,OPPONENT_ID,OPPONENT_NAME,Cluster_3pt,SEASON_YEAR,Games,Avg_FG3M_Diff,Avg_FG3M_allowed,Avg_FG3A_Diff
13,1610612738,Boston Celtics,0.0,2024-25,1,3.25,5.00,5.32
282,1610612762,Utah Jazz,0.0,2023-24,1,1.82,4.00,4.07
226,1610612757,Portland Trail Blazers,0.0,2024-25,1,1.32,3.00,1.45
283,1610612762,Utah Jazz,0.0,2024-25,1,1.25,3.00,4.28
203,1610612755,Philadelphia 76ers,0.0,2023-24,4,1.07,3.25,2.65
...,...,...,...,...,...,...,...,...
158,1610612751,Brooklyn Nets,0.0,2024-25,1,-1.76,0.00,-1.64
237,1610612758,Sacramento Kings,0.0,2023-24,1,-1.80,0.00,-0.59
248,1610612759,San Antonio Spurs,0.0,2023-24,2,-2.06,0.00,-3.35
91,1610612745,Houston Rockets,0.0,2023-24,1,-2.11,0.00,-1.78


In [676]:
opponent_ranking_3pt['as_of'] = pd.to_datetime(today)
opponent_ranking_3pt['id'] = opponent_ranking_3pt['as_of'].astype(str)+"_"+opponent_ranking_3pt['OPPONENT_ID'].astype(str)+opponent_ranking_3pt['Cluster_3pt'].astype(str)+opponent_ranking_3pt['SEASON_YEAR'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "opponent_def_3pt_rankings"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    opponent_ranking_3pt.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = opponent_ranking_3pt[~opponent_ranking_3pt['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'opponent_def_3pt_rankings' already exists. Checking for new records...
Inserted 338 new records into 'opponent_def_3pt_rankings'.


In [677]:
opponent_ranking_Ast = rolling_opp_cluster_pts.groupby(['OPPONENT_ID', 'OPPONENT_NAME','Passing_Cluster','SEASON_YEAR']).agg(Games=('GAME_DATE', 'count'), Avg_Ast_allowed=('AST', 'mean') , Avg_Ast_Diff=('AST_DIFF', 'mean')
).reset_index()

In [678]:
opponent_ranking_Ast['as_of'] = pd.to_datetime(today)
opponent_ranking_Ast['id'] = opponent_ranking_Ast['as_of'].astype(str)+"_"+opponent_ranking_Ast['OPPONENT_ID'].astype(str)+opponent_ranking_Ast['Passing_Cluster'].astype(str)+opponent_ranking_Ast['SEASON_YEAR'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "opponent_def_ast_rankings"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    opponent_ranking_Ast.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = opponent_ranking_Ast[~opponent_ranking_Ast['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'opponent_def_ast_rankings' already exists. Checking for new records...
Inserted 229 new records into 'opponent_def_ast_rankings'.


In [679]:
opponent_ranking_reb = rolling_opp_cluster_pts.groupby(['OPPONENT_ID', 'OPPONENT_NAME','Rebounding_Cluster','SEASON_YEAR']).agg(Games=('GAME_DATE', 'count'), Avg_Reb_allowed=('REB', 'mean') , Avg_Reb_Diff=('REB_DIFF', 'mean')
).reset_index()

In [680]:
opponent_ranking_reb['as_of'] = pd.to_datetime(today)
opponent_ranking_reb['id'] = opponent_ranking_reb['as_of'].astype(str)+"_"+opponent_ranking_reb['OPPONENT_ID'].astype(str)+opponent_ranking_reb['Rebounding_Cluster'].astype(str)+opponent_ranking_reb['SEASON_YEAR'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "opponent_def_reb_rankings"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    opponent_ranking_reb.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = opponent_ranking_reb[~opponent_ranking_reb['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'opponent_def_reb_rankings' already exists. Checking for new records...
Inserted 267 new records into 'opponent_def_reb_rankings'.


In [681]:
# Connect to the database
conn = sqlite3.connect('nba_data.db')

# Create a cursor object to execute SQL queries
cursor = conn.cursor()

In [682]:
table_name = "player_type_offensive"
playtype_off_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [683]:
table_name = "team_type_defensive"
teamtype_def_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [684]:
#teamtype_def_data

In [685]:
table_name = "shot_detail_data"
shotdetail_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [686]:
shot_zone_abbr = {
    'Above the Break 3': '3_AB',
    'In The Paint (Non-RA)': 'Paint',
    'Mid-Range': 'Mid',
    'Restricted Area': 'RA',
    'Left Corner 3': '3_LC',
    'Right Corner 3': '3_RC',
    'Backcourt': 'BC'
}

shotdetail_data['SHOT_ZONE_BASIC'] = shotdetail_data['SHOT_ZONE_BASIC'].map(shot_zone_abbr)

In [687]:
shotdetail_data.rename(columns={'SHOT_ATTEMPTED_FLAG':'FGA','SHOT_MADE_FLAG':'FGM'},inplace=True)

In [688]:
table_name = "drives_player_data"
drives_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [689]:
#drives_data

In [690]:
table_name = "catchshoot_player_data"
CS_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [691]:
table_name = "pullup_player_data"
PU_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [692]:
table_name = "passing_data"
passing_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [693]:
table_name = "rebounding_player_data"
rebounding_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [694]:
table_name = "speeddistance_player_data"
speeddistance_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [695]:
table_name = "closestdefender_data"
closestdefender_data = pd.read_sql(f"SELECT * FROM {table_name}", conn)

In [696]:
# 'PLAYER_ID', 'PLAYER_NAME','MIN_60G_Mavg', 'PTS_60G_Mavg', 'FGA_60G_Mavg', 'FG3A_60G_Mavg', 'MIN_60G_Msum', 
#'POSS_PCT_Cut',
#'POSS_PCT_Handoff', 'PPP_Handoff','FG_PCT_Handoff','POSS_PCT_Handoff','SHOT_FREQ_Handoff','FGA/G_Handoff',
#'POSS_PCT_Misc','SHOT_FREQ_Misc',
#'POSS_PCT_OffRebound','PPP_OffRebound','FG_PCT_OffRebound','POSS_PCT_OffRebound','SHOT_FREQ_OffRebound','FGA/G_OffRebound',
#'POSS_PCT_OffScreen', 'PPP_OffScreen', 'FG_PCT_OffScreen','POSS_PCT_OffScreen','SHOT_FREQ_OffScreen','FGA/G_OffScreen',
#'POSS_PCT_PRBallHandler', 'PPP_PRBallHandler',  'FG_PCT_PRBallHandler','POSS_PCT_PRBallHandler','SHOT_FREQ_PRBallHandler','FGA/G_PRBallHandler',
#'POSS_PCT_PRRollMan', 'PPP_PRRollMan', 'FG_PCT_PRRollMan','POSS_PCT_PRRollMan','SHOT_FREQ_PRRollMan','FGA/G_PRRollMan',
#'POSS_PCT_Spotup', 'PPP_Spotup',  'SHOT_FREQ_Spotup',
#'POSS_PCT_Transition',  'SHOT_FREQ_Transition', 
#'PPP_Cut', 'FG_PCT_Cut','POSS_PCT_Cut','SHOT_FREQ_Cut','FGA/G_Cut',
#'PPP_Isolation', 'FG_PCT_Isolation','POSS_PCT_Isolation','SHOT_FREQ_Isolation','FGA/G_Isolation',
#'PPP_Postup', 'FG_PCT_Postup','POSS_PCT_Postup','SHOT_FREQ_Postup','FGA/G_Postup',
#'AVG_SPEED_OFF_60G_Msum', 'OFFmiles_PER_MIN',
#'PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day','PU_FGA_PCT_60Day', 'PU_FGA_rate_60Day','PU_FG3A_rate_60Day', 
#'CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day','CS_FGA_rate_60Day', 'CS_FG3A_rate_60Day',
#'PTS_PER_DRIVE', 'DRIVES/MIN_60Day','SHOTS_DRIVE_RATE',
#'FGA_3_AB_60G_Msavg', 'FGA_3_LC_60G_Msavg','FGA_3_RC_60G_Msavg', 'FGA_BC_60G_Msavg', 'FGA_Mid_60G_Msavg',
#'FGA_Paint_60G_Msavg', 'FGA_RA_60G_Msavg', 'FGM_3_AB_60G_Msavg','FGM_3_LC_60G_Msavg', 'FGM_3_RC_60G_Msavg', 'FGM_BC_60G_Msavg',
#'FGM_Mid_60G_Msavg', 'FGM_Paint_60G_Msavg', 'FGM_RA_60G_Msavg']]

In [697]:
matchups = rolling_opp_gamelogs[['PLAYER_ID','PLAYER_NAME','GAME_DATE','OPPONENT_ID', 'OPPONENT_NAME','Cluster_Pts','Cluster_3pt','GAMES_IN_WINDOW_60G']]

In [698]:
matchups

,PLAYER_ID,PLAYER_NAME,GAME_DATE,OPPONENT_ID,OPPONENT_NAME,Cluster_Pts,Cluster_3pt,GAMES_IN_WINDOW_60G
0,2544,LeBron James,2023-10-24,1610612743,Denver Nuggets,5.0,2.0,1.0
1,2544,LeBron James,2023-10-26,1610612756,Phoenix Suns,5.0,2.0,2.0
2,2544,LeBron James,2023-10-29,1610612758,Sacramento Kings,5.0,2.0,3.0
3,2544,LeBron James,2023-10-30,1610612753,Orlando Magic,5.0,2.0,4.0
4,2544,LeBron James,2023-11-01,1610612746,LA Clippers,5.0,2.0,5.0
...,...,...,...,...,...,...,...,...
47086,1642530,Yuki Kawamura,2024-12-29,1610612760,Oklahoma City Thunder,NaN,NaN,15.0
47087,1642530,Yuki Kawamura,2025-01-06,1610612742,Dallas Mavericks,NaN,NaN,16.0
47088,1642530,Yuki Kawamura,2025-01-17,1610612759,San Antonio Spurs,NaN,NaN,17.0
47089,1642530,Yuki Kawamura,2025-02-05,1610612761,Toronto Raptors,NaN,NaN,18.0


In [699]:
#drives_sorted = pd.DataFrame(drives_data.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [700]:
gamelogs_drives= calculate_weighted_rolling_stats(
    gamelogs=drives_data,
    rolling_columns=['DRIVES','DRIVE_FGA','DRIVE_PASSES','DRIVE_AST','DRIVE_PTS'],
    method = 'average'
)

In [701]:
drives_data['GAME_DATE'] = pd.to_datetime(drives_data['GAME_DATE'])
drives_data_def = matchups.merge(drives_data, how='left')

In [702]:
gamelogs_drives_team= calculate_team_rolling_stats(
    gamelogs=drives_data_def,
    rolling_columns=['DRIVES','DRIVE_FGA','DRIVE_PASSES','DRIVE_AST','DRIVE_PTS'],
    method = 'sum'
)

In [703]:
gamelogs_drives_team_ = pd.DataFrame(gamelogs_drives_team[['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Cluster_Pts', 'Cluster_3pt', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'MIN', 'DRIVES', 'DRIVE_FGM',
       'DRIVE_FGA', 'DRIVE_FG_PCT', 'DRIVE_FTM', 'DRIVE_FTA', 'DRIVE_FT_PCT',
       'DRIVE_PTS', 'DRIVE_PTS_PCT', 'DRIVE_PASSES', 'DRIVE_PASSES_PCT',
       'DRIVE_AST', 'DRIVE_AST_PCT', 'DRIVE_TOV', 'SEASON_YEAR',
       'GAMES_IN_WINDOW_TEAM_60G', 'DRIVES_60G_Sum', 'DRIVE_FGA_60G_Sum',
       'DRIVE_PASSES_60G_Sum', 'DRIVE_AST_60G_Sum', 'DRIVE_PTS_60G_Sum']])

In [704]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
gamelogs_drives_team_['id'] = gamelogs_drives_team_['GAME_DATE'].astype(str)+"_"+gamelogs_drives_team_['PLAYER_ID'].astype(str)+"_"+gamelogs_drives_team_['TEAM_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "drives_team_def_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelogs_drives_team_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelogs_drives_team_[~gamelogs_drives_team_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'drives_team_def_rolling' already exists. Checking for new records...
Inserted 1181 new records into 'drives_team_def_rolling'.


In [705]:
#drives_data_def

In [706]:
#drives = pd.DataFrame(gamelogs_drives[['PLAYER_ID','GAME_DATE','GAMES_IN_WINDOW_60G','DRIVES_60G_WAvg', 'DRIVE_FGA_60G_WAvg', 'DRIVE_PASSES_60G_WAvg','DRIVE_AST_60G_WAvg', 'DRIVE_PTS_60G_WAvg']])

In [707]:
#CS_sorted = pd.DataFrame(CS_data.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [708]:
gamelogs_cs = calculate_weighted_rolling_stats(
    gamelogs=CS_data,
    rolling_columns=['CATCH_SHOOT_FGM','CATCH_SHOOT_FGA','CATCH_SHOOT_FG3M','CATCH_SHOOT_FG3A'],
    method = 'average'
)

In [709]:
CS_data['GAME_DATE'] = pd.to_datetime(CS_data['GAME_DATE'])
CS_data_def = matchups.merge(CS_data, how='left')

In [710]:
gamelogs_cs_team = calculate_team_rolling_stats(
    gamelogs=CS_data_def,
    rolling_columns=['CATCH_SHOOT_FGM','CATCH_SHOOT_FGA','CATCH_SHOOT_FG3M','CATCH_SHOOT_FG3A'],
    method = 'sum'
)

In [711]:
gamelogs_cs_team_ = pd.DataFrame(gamelogs_cs_team[['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Cluster_Pts', 'Cluster_3pt', 'GAMES_IN_WINDOW_60G', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'MIN', 'CATCH_SHOOT_FGM',
       'CATCH_SHOOT_FGA', 'CATCH_SHOOT_FG_PCT', 'CATCH_SHOOT_PTS',
       'CATCH_SHOOT_FG3M', 'CATCH_SHOOT_FG3A', 'CATCH_SHOOT_FG3_PCT', 'SEASON_YEAR',
       'GAMES_IN_WINDOW_TEAM_60G', 'CATCH_SHOOT_FGM_60G_Sum',
       'CATCH_SHOOT_FGA_60G_Sum', 'CATCH_SHOOT_FG3M_60G_Sum',
       'CATCH_SHOOT_FG3A_60G_Sum']])

In [712]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
gamelogs_cs_team_['id'] = gamelogs_cs_team_['GAME_DATE'].astype(str)+"_"+gamelogs_cs_team_['PLAYER_ID'].astype(str)+"_"+gamelogs_cs_team_['TEAM_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "catchshoot_team_def_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelogs_cs_team_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelogs_cs_team_[~gamelogs_cs_team_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'catchshoot_team_def_rolling' already exists. Checking for new records...
Inserted 1181 new records into 'catchshoot_team_def_rolling'.


In [713]:
#catch_shoot = pd.DataFrame(gamelogs_cs[['PLAYER_ID','GAME_DATE', 'GAMES_IN_WINDOW_60G', 'CATCH_SHOOT_FGM_60G_WSum', 'CATCH_SHOOT_FGA_60G_WSum', 'CATCH_SHOOT_FG3M_60G_WSum','CATCH_SHOOT_FG3A_60G_WSum']])

In [714]:
#PU_sorted = pd.DataFrame(PU_data.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [715]:
gamelogs_pu = calculate_weighted_rolling_stats(
    gamelogs=PU_data,
    rolling_columns=['PULL_UP_FGM','PULL_UP_FGA','PULL_UP_FG3M','PULL_UP_FG3A'],
    method = 'average'
)

In [716]:
PU_data['GAME_DATE'] = pd.to_datetime(PU_data['GAME_DATE'])
PU_data_def = matchups.merge(PU_data, how='left')

In [717]:
gamelogs_pu_team = calculate_team_rolling_stats(
    gamelogs=PU_data_def,
    rolling_columns=['PULL_UP_FGM','PULL_UP_FGA','PULL_UP_FG3M','PULL_UP_FG3A'],
    method = 'sum'
)

In [718]:
gamelogs_pu_team_ = pd.DataFrame(gamelogs_pu_team[['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Cluster_Pts', 'Cluster_3pt', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'MIN', 'PULL_UP_FGM',
       'PULL_UP_FGA', 'PULL_UP_FG_PCT', 'PULL_UP_PTS', 'PULL_UP_FG3M',
       'PULL_UP_FG3A', 'PULL_UP_FG3_PCT', 'PULL_UP_EFG_PCT', 'GAMES_IN_WINDOW_TEAM_60G', 'PULL_UP_FGM_60G_Sum',
       'PULL_UP_FGA_60G_Sum', 'PULL_UP_FG3M_60G_Sum', 'PULL_UP_FG3A_60G_Sum']])

In [719]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
gamelogs_pu_team_['id'] = gamelogs_pu_team_['GAME_DATE'].astype(str)+"_"+gamelogs_pu_team_['PLAYER_ID'].astype(str)+"_"+gamelogs_pu_team_['TEAM_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "pullup_team_def_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    gamelogs_pu_team_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = gamelogs_pu_team_[~gamelogs_pu_team_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'pullup_team_def_rolling' already exists. Checking for new records...
Inserted 1181 new records into 'pullup_team_def_rolling'.


In [720]:
#pullup = pd.DataFrame(gamelogs_pu[['PLAYER_ID', 'GAME_DATE','GAMES_IN_WINDOW_60G', 'PULL_UP_FGM_60G_WSum', 'PULL_UP_FGA_60G_WSum','PULL_UP_FG3M_60G_WSum', 'PULL_UP_FG3A_60G_WSum']])

In [721]:
shotdetail_sum= pd.DataFrame(shotdetail_data.groupby(['SEASON_YEAR',"PLAYER_ID",'GAME_DATE','SHOT_ZONE_BASIC'])[['FGA','FGM']].sum()).reset_index()

In [722]:
shotdetail_data['GAME_DATE'] = pd.to_datetime(shotdetail_data['GAME_DATE'])


In [723]:
shotdetail_data_def = matchups.merge(shotdetail_data, how='left')

In [724]:
shotdetail_sum_def = pd.DataFrame(shotdetail_data_def.groupby(['SEASON_YEAR',"OPPONENT_ID",'GAME_DATE','SHOT_ZONE_BASIC'])[['FGA','FGM']].sum()).reset_index()
shotdetail_sum_def_pts = pd.DataFrame(shotdetail_data_def.groupby(['SEASON_YEAR',"OPPONENT_ID",'Cluster_Pts','GAME_DATE','SHOT_ZONE_BASIC'])[['FGA','FGM']].sum()).reset_index()
shotdetail_sum_def_3s = pd.DataFrame(shotdetail_data_def.groupby(['SEASON_YEAR',"OPPONENT_ID",'Cluster_3pt','GAME_DATE','SHOT_ZONE_BASIC'])[['FGA','FGM']].sum()).reset_index()

In [725]:
#shotdetail_sum_def_pts

In [726]:
#shotdetail_data.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=True)
shotdetail_pivot = shotdetail_sum.pivot(index=['SEASON_YEAR','PLAYER_ID','GAME_DATE'], 
                    columns='SHOT_ZONE_BASIC', 
                    values=['FGA','FGM', ])

shotdetail_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in shotdetail_pivot.columns]
shotdetail_pivot.reset_index(inplace=True)

In [727]:
#shotdetail_data.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=True)
shotdetail_pivot_def = shotdetail_sum_def.pivot(index=['SEASON_YEAR','OPPONENT_ID','GAME_DATE'], 
                    columns='SHOT_ZONE_BASIC', 
                    values=['FGA','FGM', ])

shotdetail_pivot_def.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in shotdetail_pivot_def.columns]
shotdetail_pivot_def.reset_index(inplace=True)

#Def Pts
shotdetail_pivot_def_pts = shotdetail_sum_def_pts.pivot(index=['SEASON_YEAR','OPPONENT_ID','Cluster_Pts','GAME_DATE'], 
                    columns='SHOT_ZONE_BASIC', 
                    values=['FGA','FGM', ])

shotdetail_pivot_def_pts.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in shotdetail_pivot_def_pts.columns]
shotdetail_pivot_def_pts.reset_index(inplace=True)

#3pt Def
shotdetail_pivot_def_3pt = shotdetail_sum_def_3s.pivot(index=['SEASON_YEAR','OPPONENT_ID','Cluster_3pt','GAME_DATE'], 
                    columns='SHOT_ZONE_BASIC', 
                    values=['FGA','FGM', ])

shotdetail_pivot_def_3pt.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in shotdetail_pivot_def_3pt.columns]
shotdetail_pivot_def_3pt.reset_index(inplace=True)

In [728]:
shotdetail_pivot_def_pts.columns

Index(['SEASON_YEAR', 'OPPONENT_ID', 'Cluster_Pts', 'GAME_DATE', 'FGA_3_AB',
       'FGA_3_LC', 'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA',
       'FGM_3_AB', 'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint',
       'FGM_RA'],
      dtype='object')

In [729]:
gamelogs_sd_team_pt = calculate_team_rolling_stats(
    gamelogs=shotdetail_pivot_def_pts,
    rolling_columns=['FGA_3_AB',
       'FGA_3_LC', 'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA',
       'FGM_3_AB', 'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint','FGM_RA'],
    group_columns = ['OPPONENT_ID','Cluster_Pts'],
    method = 'sum'
)

gamelogs_sd_team_pt_avg = calculate_team_rolling_stats(
    gamelogs=shotdetail_pivot_def_pts,
    rolling_columns=['FGA_3_AB',
       'FGA_3_LC', 'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA',
       'FGM_3_AB', 'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint','FGM_RA'],
    group_columns = ['OPPONENT_ID','Cluster_Pts'],
    method = 'average'
)

In [730]:
gamelogs_sd_team_pt['GAME_DATE'] = pd.to_datetime(gamelogs_sd_team_pt['GAME_DATE'])
gamelogs_sd_team_pt_avg['GAME_DATE'] = pd.to_datetime(gamelogs_sd_team_pt_avg['GAME_DATE'])

gl_cluster_shotdetail_def = gamelogs_sd_team_pt.merge(gamelogs_sd_team_pt_avg,)


In [731]:
gamelogs_def_shotdetail_sum = calculate_team_rolling_stats(
    gamelogs=shotdetail_pivot_def,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'sum'
)
gamelogs_def_shotdetail_avg = calculate_team_rolling_stats(
    gamelogs=shotdetail_pivot_def,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'average'
)

In [732]:
gamelogs_def_shotdetail_sum['GAME_DATE'] = pd.to_datetime(gamelogs_def_shotdetail_sum['GAME_DATE'])
gamelogs_def_shotdetail_avg['GAME_DATE'] = pd.to_datetime(gamelogs_def_shotdetail_avg['GAME_DATE'])

In [733]:
gamelogs_def_shotdetail = gamelogs_def_shotdetail_sum.merge(gamelogs_def_shotdetail_avg, how='left')

In [734]:
#shotdetail_sorted = pd.DataFrame(shotdetail_pivot.sort_values(by=['PLAYER_ID','GAME_DATE'])).reset_index(drop=True)

In [735]:
gamelogs_shotdetail_sum = calculate_weighted_rolling_stats(
    gamelogs=shotdetail_pivot,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'sum'
)

In [736]:
gamelogs_shotdetail_avg = calculate_weighted_rolling_stats(
    gamelogs=shotdetail_pivot,
    rolling_columns=['FGA_3_AB', 'FGA_3_LC',
       'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint', 'FGA_RA', 'FGM_3_AB',
       'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid', 'FGM_Paint', 'FGM_RA'],
    method = 'average'
)

In [737]:
gamelogs_shotdetail_sum['GAME_DATE'] = pd.to_datetime(gamelogs_shotdetail_sum['GAME_DATE'])
gamelogs_shotdetail_avg['GAME_DATE'] = pd.to_datetime(gamelogs_shotdetail_avg['GAME_DATE'])

In [738]:
gamelogs_shotdetail = gamelogs_shotdetail_sum.merge(gamelogs_shotdetail_avg, how='left')

In [739]:
#gamelogs_shotdetail

In [740]:
closestdefender_data['CLOSEST_DEFENDER_DISTANCE_RANGE'].unique()

array(['0-2 Feet - Very Tight', '2-4 Feet - Tight', '4-6 Feet - Open',
       '6+ Feet - Wide Open'], dtype=object)

In [741]:
CLOSEST_DEFENDER = {
    '0-2 Feet - Very Tight': 'Very_Tight',
    '2-4 Feet - Tight': 'Tight',
    '4-6 Feet - Open': 'Open',
    '6+ Feet - Wide Open': 'Wide_Open',
}

In [742]:
closestdefender_data['CLOSEST_DEFENDER_DISTANCE_RANGE'] = closestdefender_data['CLOSEST_DEFENDER_DISTANCE_RANGE'].map(CLOSEST_DEFENDER)

In [743]:
#shotdetail_data.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=True)
closestdefender_pivot = closestdefender_data.pivot(index=['SEASON_YEAR','PLAYER_NAME','PLAYER_ID','GAME_DATE'], 
                    columns='CLOSEST_DEFENDER_DISTANCE_RANGE', 
                    values=['FGA','FGM', 'FG3A','FG3M'])

closestdefender_pivot.columns = ['_'.join(col).strip() if isinstance(col, tuple) else col for col in closestdefender_pivot.columns]
closestdefender_pivot.reset_index(inplace=True)

In [744]:
closestdefender_pivot.fillna(0, inplace=True)

In [745]:

gamelogs_closestdefender_avg = calculate_weighted_rolling_stats(
    gamelogs=closestdefender_pivot,
    rolling_columns=['FGA_Open', 'FGA_Tight',
       'FGA_Very_Tight', 'FGA_Wide_Open', 'FGM_Open', 'FGM_Tight',
       'FGM_Very_Tight', 'FGM_Wide_Open', 'FG3A_Open', 'FG3A_Tight',
       'FG3A_Very_Tight', 'FG3A_Wide_Open', 'FG3M_Open', 'FG3M_Tight',
       'FG3M_Very_Tight', 'FG3M_Wide_Open'],
    method = 'average'
)

gamelogs_closestdefender_sum = calculate_weighted_rolling_stats(
    gamelogs=closestdefender_pivot,
    rolling_columns=['FGA_Open', 'FGA_Tight',
       'FGA_Very_Tight', 'FGA_Wide_Open', 'FGM_Open', 'FGM_Tight',
       'FGM_Very_Tight', 'FGM_Wide_Open', 'FG3A_Open', 'FG3A_Tight',
       'FG3A_Very_Tight', 'FG3A_Wide_Open', 'FG3M_Open', 'FG3M_Tight',
       'FG3M_Very_Tight', 'FG3M_Wide_Open'],
    method = 'sum'
)

In [746]:
gamelogs_closestdefender_sum['GAME_DATE'] = pd.to_datetime(gamelogs_closestdefender_sum['GAME_DATE'])
gamelogs_closestdefender_avg['GAME_DATE'] = pd.to_datetime(gamelogs_closestdefender_avg['GAME_DATE'])

In [747]:
closestdefender_60Day = gamelogs_closestdefender_sum.merge(gamelogs_closestdefender_avg, how='left')

In [748]:
closestdefender_60Day_log = matchups.merge(closestdefender_60Day[['SEASON_YEAR', 'PLAYER_NAME', 'PLAYER_ID', 'GAME_DATE', 'FGA_Open',
       'FGA_Tight', 'FGA_Very_Tight', 'FGA_Wide_Open', 'FGM_Open', 'FGM_Tight',
       'FGM_Very_Tight', 'FGM_Wide_Open', 'FG3A_Open', 'FG3A_Tight',
       'FG3A_Very_Tight', 'FG3A_Wide_Open', 'FG3M_Open', 'FG3M_Tight',
       'FG3M_Very_Tight', 'FG3M_Wide_Open',
       'FGA_Open_60G_Sum', 'FGA_Tight_60G_Sum', 'FGA_Very_Tight_60G_Sum',
       'FGA_Wide_Open_60G_Sum', 'FGM_Open_60G_Sum', 'FGM_Tight_60G_Sum',
       'FGM_Very_Tight_60G_Sum', 'FGM_Wide_Open_60G_Sum', 'FG3A_Open_60G_Sum',
       'FG3A_Tight_60G_Sum', 'FG3A_Very_Tight_60G_Sum',
       'FG3A_Wide_Open_60G_Sum', 'FG3M_Open_60G_Sum', 'FG3M_Tight_60G_Sum',
       'FG3M_Very_Tight_60G_Sum', 'FG3M_Wide_Open_60G_Sum',
       'FGA_Open_60G_Mavg', 'FGA_Tight_60G_Mavg', 'FGA_Very_Tight_60G_Mavg',
       'FGA_Wide_Open_60G_Mavg', 'FGM_Open_60G_Mavg', 'FGM_Tight_60G_Mavg',
       'FGM_Very_Tight_60G_Mavg', 'FGM_Wide_Open_60G_Mavg',
       'FG3A_Open_60G_Mavg', 'FG3A_Tight_60G_Mavg', 'FG3A_Very_Tight_60G_Mavg',
       'FG3A_Wide_Open_60G_Mavg', 'FG3M_Open_60G_Mavg', 'FG3M_Tight_60G_Mavg',
       'FG3M_Very_Tight_60G_Mavg', 'FG3M_Wide_Open_60G_Mavg']], how='left')

In [749]:
#gamelogs_shotdetail_60Day['as_of'] = pd.to_datetime(today)
closestdefender_60Day_log['id'] = closestdefender_60Day_log['GAME_DATE'].astype(str)+"_"+closestdefender_60Day_log['PLAYER_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "closestdef_player_rolling"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    closestdefender_60Day_log.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = closestdefender_60Day_log[~closestdefender_60Day_log['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

Table 'closestdef_player_rolling' already exists. Checking for new records...
Inserted 1133 new records into 'closestdef_player_rolling'.


In [750]:
closestdefender_60Day_sum = pd.DataFrame(closestdefender_60Day[['SEASON_YEAR', 'PLAYER_NAME', 'PLAYER_ID', 'GAME_DATE', 'FGA_Open',
       'FGA_Tight', 'FGA_Very_Tight', 'FGA_Wide_Open', 'FGM_Open', 'FGM_Tight',
       'FGM_Very_Tight', 'FGM_Wide_Open', 'FG3A_Open', 'FG3A_Tight',
       'FG3A_Very_Tight', 'FG3A_Wide_Open', 'FG3M_Open', 'FG3M_Tight',
       'FG3M_Very_Tight', 'FG3M_Wide_Open', 'FGA_Open_60G_Sum',
       'FGA_Tight_60G_Sum', 'FGA_Very_Tight_60G_Sum',
       'FGA_Wide_Open_60G_Sum', 'FGM_Open_60G_Sum', 'FGM_Tight_60G_Sum',
       'FGM_Very_Tight_60G_Sum', 'FGM_Wide_Open_60G_Sum',
       'FG3A_Open_60G_Sum', 'FG3A_Tight_60G_Sum', 'FG3A_Very_Tight_60G_Sum',
       'FG3A_Wide_Open_60G_Sum', 'FG3M_Open_60G_Sum', 'FG3M_Tight_60G_Sum',
       'FG3M_Very_Tight_60G_Sum', 'FG3M_Wide_Open_60G_Sum',
       'FGA_Open_60G_Mavg', 'FGA_Tight_60G_Mavg', 'FGA_Very_Tight_60G_Mavg',
       'FGA_Wide_Open_60G_Mavg', 'FGM_Open_60G_Mavg', 'FGM_Tight_60G_Mavg',
       'FGM_Very_Tight_60G_Mavg', 'FGM_Wide_Open_60G_Mavg',
       'FG3A_Open_60G_Mavg', 'FG3A_Tight_60G_Mavg', 'FG3A_Very_Tight_60G_Mavg',
       'FG3A_Wide_Open_60G_Mavg', 'FG3M_Open_60G_Mavg', 'FG3M_Tight_60G_Mavg',
       'FG3M_Very_Tight_60G_Mavg', 'FG3M_Wide_Open_60G_Mavg']])

In [751]:
closestdefender_60Day_sum['FG3A_Open_diff'] = closestdefender_60Day_sum['FG3A_Open'] - closestdefender_60Day_sum['FG3A_Open_60G_Mavg']
closestdefender_60Day_sum['FGA_Open_diff'] = closestdefender_60Day_sum['FGA_Open'] - closestdefender_60Day_sum['FGA_Open_60G_Mavg']
closestdefender_60Day_sum['FG3A_Wide_Open_diff'] = closestdefender_60Day_sum['FG3A_Wide_Open'] - closestdefender_60Day_sum['FG3A_Wide_Open_60G_Mavg']
closestdefender_60Day_sum['FGA_Wide_Open_diff'] = closestdefender_60Day_sum['FGA_Wide_Open'] - closestdefender_60Day_sum['FGA_Wide_Open_60G_Mavg']
closestdefender_60Day_sum['FG3A_Tight_diff'] = closestdefender_60Day_sum['FG3A_Tight'] - closestdefender_60Day_sum['FG3A_Tight_60G_Mavg']
closestdefender_60Day_sum['FGA_Tight_diff'] = closestdefender_60Day_sum['FGA_Tight'] - closestdefender_60Day_sum['FGA_Tight_60G_Mavg']
closestdefender_60Day_sum['FG3A_Very_Tight_diff'] = closestdefender_60Day_sum['FG3A_Very_Tight'] - closestdefender_60Day_sum['FG3A_Very_Tight_60G_Mavg']
closestdefender_60Day_sum['FGA_Very_Tight_diff'] = closestdefender_60Day_sum['FGA_Very_Tight'] - closestdefender_60Day_sum['FGA_Very_Tight_60G_Mavg']

In [752]:
#rolling_opp_gamelogs

In [753]:
#closestdefender_60Day_sum

In [754]:
matchups = rolling_opp_gamelogs[['PLAYER_ID','PLAYER_NAME','GAME_DATE','OPPONENT_ID', 'OPPONENT_NAME','Cluster_Pts','Cluster_3pt','GAMES_IN_WINDOW_60G']]

In [755]:
closestdefender_60Day_sum

,SEASON_YEAR,PLAYER_NAME,PLAYER_ID,GAME_DATE,FGA_Open,FGA_Tight,FGA_Very_Tight,FGA_Wide_Open,FGM_Open,FGM_Tight,FGM_Very_Tight,FGM_Wide_Open,FG3A_Open,FG3A_Tight,FG3A_Very_Tight,FG3A_Wide_Open,FG3M_Open,FG3M_Tight,FG3M_Very_Tight,FG3M_Wide_Open,FGA_Open_60G_Sum,FGA_Tight_60G_Sum,FGA_Very_Tight_60G_Sum,FGA_Wide_Open_60G_Sum,FGM_Open_60G_Sum,FGM_Tight_60G_Sum,FGM_Very_Tight_60G_Sum,FGM_Wide_Open_60G_Sum,FG3A_Open_60G_Sum,FG3A_Tight_60G_Sum,FG3A_Very_Tight_60G_Sum,FG3A_Wide_Open_60G_Sum,FG3M_Open_60G_Sum,FG3M_Tight_60G_Sum,FG3M_Very_Tight_60G_Sum,FG3M_Wide_Open_60G_Sum,FGA_Open_60G_Mavg,FGA_Tight_60G_Mavg,FGA_Very_Tight_60G_Mavg,FGA_Wide_Open_60G_Mavg,FGM_Open_60G_Mavg,FGM_Tight_60G_Mavg,FGM_Very_Tight_60G_Mavg,FGM_Wide_Open_60G_Mavg,FG3A_Open_60G_Mavg,FG3A_Tight_60G_Mavg,FG3A_Very_Tight_60G_Mavg,FG3A_Wide_Open_60G_Mavg,FG3M_Open_60G_Mavg,FG3M_Tight_60G_Mavg,FG3M_Very_Tight_60G_Mavg,FG3M_Wide_Open_60G_Mavg,FG3A_Open_diff,FGA_Open_diff,FG3A_Wide_Open_diff,FGA_Wide_Open_diff,FG3A_Tight_diff,FGA_Tight_diff,FG3A_Very_Tight_diff,FGA_Very_Tight_diff
0,2023-24,LeBron James,2544,2023-10-24,5.0,7.0,3.0,1.0,3.0,4.0,3.0,0.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,5.0,7.0,3.0,1.0,3.0,4.0,3.0,0.0,2.0,1.0,0.0,1.0,1.0,0.0,0.0,0.0,5.000000,7.000000,3.0,1.000000,3.000000,4.000000,3.00,0.000000,2.000000,1.000000,0.0,1.000000,1.000000,0.00,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
1,2023-24,LeBron James,2544,2023-10-26,4.0,10.0,0.0,0.0,2.0,5.0,0.0,0.0,3.0,2.0,0.0,0.0,1.0,0.0,0.0,0.0,9.0,17.0,3.0,1.0,5.0,9.0,3.0,0.0,5.0,3.0,0.0,1.0,2.0,0.0,0.0,0.0,4.500000,8.500000,1.5,0.500000,2.500000,4.500000,1.50,0.000000,2.500000,1.500000,0.0,0.500000,1.000000,0.00,0.0,0.000000,0.500000,-0.500000,-0.500000,-0.500000,0.500000,1.500000,0.0,-1.5
2,2023-24,LeBron James,2544,2023-10-29,8.0,8.0,0.0,3.0,3.0,5.0,0.0,3.0,6.0,1.0,0.0,1.0,2.0,0.0,0.0,1.0,17.0,25.0,3.0,4.0,8.0,14.0,3.0,3.0,11.0,4.0,0.0,2.0,4.0,0.0,0.0,1.0,5.666667,8.333333,1.0,1.333333,2.666667,4.666667,1.00,1.000000,3.666667,1.333333,0.0,0.666667,1.333333,0.00,0.0,0.333333,2.333333,2.333333,0.333333,1.666667,-0.333333,-0.333333,0.0,-1.0
3,2023-24,LeBron James,2544,2023-10-30,8.0,6.0,1.0,2.0,1.0,5.0,0.0,1.0,5.0,1.0,0.0,0.0,1.0,1.0,0.0,0.0,25.0,31.0,4.0,6.0,9.0,19.0,3.0,4.0,16.0,5.0,0.0,2.0,5.0,1.0,0.0,1.0,6.250000,7.750000,1.0,1.500000,2.250000,4.750000,0.75,1.000000,4.000000,1.250000,0.0,0.500000,1.250000,0.25,0.0,0.250000,1.000000,1.750000,-0.500000,0.500000,-0.250000,-1.750000,0.0,0.0
4,2023-24,LeBron James,2544,2023-11-01,6.0,7.0,3.0,3.0,4.0,5.0,2.0,2.0,4.0,3.0,0.0,1.0,3.0,1.0,0.0,0.0,31.0,38.0,7.0,9.0,13.0,24.0,5.0,6.0,20.0,8.0,0.0,3.0,8.0,2.0,0.0,1.0,6.200000,7.600000,1.4,1.800000,2.600000,4.800000,1.00,1.200000,4.000000,1.600000,0.0,0.600000,1.600000,0.40,0.0,0.200000,0.000000,-0.200000,0.400000,1.200000,1.400000,-0.600000,0.0,1.6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
44276,2024-25,Yuki Kawamura,1642530,2024-11-27,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0,2.0,0.0,3.0,2.0,1.0,0.0,0.0,6.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,1.000000,0.250000,0.0,0.375000,0.250000,0.125000,0.00,0.000000,0.750000,0.000000,0.0,0.375000,0.125000,0.00,0.0,0.000000,-0.750000,-1.000000,-0.375000,-0.375000,0.000000,0.750000,0.0,0.0
44277,2024-25,Yuki Kawamura,1642530,2024-12-08,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,2.0,0.0,3.0,2.0,1.0,0.0,0.0,7.0,0.0,0.0,3.0,1.0,0.0,0.0,0.0,1.000000,0.222222,0.0,0.333333,0.222222,0.111111,0.00,0.000000,0.777778,0.000000,0.0,0.333333,0.111111,0.00,0.0,0.000000,0.222222,0.000000,-0.333333,-0.333333,0.000000,-0.222222,0.0,0.0
44278,2024-25,Yuki Kawamura,1642530,2024-12-29,3.0,1.0,0.0,1.0,2.0,1.0,0.0,1.0,3.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,12.0,3.0,0.0,4.0,4.0,2.0,0.0,1.0,10.0,0.0,0.0,3.0,3.0,0.0,0.0,0.0,1.200000,0.300000,0.0,

In [756]:
closestdefender_matchup = matchups.merge(closestdefender_60Day_sum, how='left')

In [757]:
closest_fg_rev = closestdefender_matchup.groupby(['SEASON_YEAR'])[['FGA_Open', 'FGA_Tight', 'FGA_Very_Tight', 'FGA_Wide_Open', 'FGM_Open',
       'FGM_Tight', 'FGM_Very_Tight', 'FGM_Wide_Open', 'FG3A_Open',
       'FG3A_Tight', 'FG3A_Very_Tight', 'FG3A_Wide_Open', 'FG3M_Open',
       'FG3M_Tight', 'FG3M_Very_Tight', 'FG3M_Wide_Open']].sum().reset_index()

In [758]:
pd.DataFrame(closest_fg_rev[['SEASON_YEAR', 'FGA_Open','FGM_Open','FGA_Tight','FGM_Tight','FGA_Very_Tight','FGM_Very_Tight', 'FGA_Wide_Open','FGM_Wide_Open', 
                'FG3A_Open','FG3M_Open','FG3A_Tight', 'FG3M_Tight','FG3A_Very_Tight','FG3M_Very_Tight','FG3A_Wide_Open','FG3M_Wide_Open']])

,SEASON_YEAR,FGA_Open,FGM_Open,FGA_Tight,FGM_Tight,FGA_Very_Tight,FGM_Very_Tight,FGA_Wide_Open,FGM_Wide_Open,FG3A_Open,FG3M_Open,FG3A_Tight,FG3M_Tight,FG3A_Very_Tight,FG3M_Very_Tight,FG3A_Wide_Open,FG3M_Wide_Open
0,2023-24,60115.0,27339.0,82010.0,42529.0,21691.0,10191.0,52224.0,22413.0,31417.0,11008.0,8890.0,2635.0,399.0,108.0,44611.0,17458.0
1,2024-25,50787.0,22403.0,65395.0,33562.0,11469.0,5238.0,40865.0,17286.0,27821.0,9507.0,7243.0,2128.0,315.0,90.0,35382.0,13654.0


In [759]:
closest_fg_rev = pd.DataFrame(closest_fg_rev[['SEASON_YEAR', 'FGA_Open','FGM_Open','FGA_Tight','FGM_Tight','FGA_Very_Tight','FGM_Very_Tight', 'FGA_Wide_Open','FGM_Wide_Open', 
                'FG3A_Open','FG3M_Open','FG3A_Tight', 'FG3M_Tight','FG3A_Very_Tight','FG3M_Very_Tight','FG3A_Wide_Open','FG3M_Wide_Open']])
closest_fg_rev['FG_WIDEOPEN%'] =((closest_fg_rev['FGM_Wide_Open'] /closest_fg_rev['FGA_Wide_Open'])*100).round(2)
closest_fg_rev['FG_OPEN%']=((closest_fg_rev['FGM_Open'] /closest_fg_rev['FGA_Open'])*100).round(2)
closest_fg_rev['FG_TIGHT%']=((closest_fg_rev['FGM_Tight'] /closest_fg_rev['FGA_Tight'])*100).round(2)
closest_fg_rev['FG_VERYTIGHT%']=((closest_fg_rev['FGM_Very_Tight'] /closest_fg_rev['FGA_Very_Tight'])*100).round(2)
closest_fg_rev['FG3_WIDEOPEN%'] =((closest_fg_rev['FG3M_Wide_Open'] /closest_fg_rev['FG3A_Wide_Open'])*100).round(2)
closest_fg_rev['FG3_OPEN%']=((closest_fg_rev['FG3M_Open'] /closest_fg_rev['FG3A_Open'])*100).round(2)
closest_fg_rev['FG3_TIGHT%']=((closest_fg_rev['FG3M_Tight'] /closest_fg_rev['FG3A_Tight'])*100).round(2)
closest_fg_rev['F3G_VERYTIGHT%']=((closest_fg_rev['FG3M_Very_Tight'] /closest_fg_rev['FG3A_Very_Tight'])*100).round(2)

closest_fg_rev[['SEASON_YEAR','FG_WIDEOPEN%','FG_OPEN%','FG_TIGHT%','FG_VERYTIGHT%',
               'FG3_WIDEOPEN%','FG3_OPEN%','FG3_TIGHT%','F3G_VERYTIGHT%']]

,SEASON_YEAR,FG_WIDEOPEN%,FG_OPEN%,FG_TIGHT%,FG_VERYTIGHT%,FG3_WIDEOPEN%,FG3_OPEN%,FG3_TIGHT%,F3G_VERYTIGHT%
0,2023-24,42.92,45.48,51.86,46.98,39.13,35.04,29.64,27.07
1,2024-25,42.30,44.11,51.32,45.67,38.59,34.17,29.38,28.57


In [760]:
#shotdetail_data.rename(columns = {'SHOT_ATTEMPTED_FLAG_60G_Msum':'FGA_60G_Msum','SHOT_MADE_FLAG_60G_Msum':'FGM_60G_Msum','SHOT_ATTEMPTED_FLAG_60G_Msavg':'FGA_60G_Mavg','SHOT_MADE_FLAG_60G_Msavg':'FGM_60G_Mavg'}, inplace=Tru

In [761]:
#closestdefender_matchup.fillna(0, )

In [762]:
 # Group by the requested columns and calculate metrics
grouped_def = closestdefender_matchup.groupby([
    'OPPONENT_ID',
    "SEASON_YEAR",
    'OPPONENT_NAME',
    'Cluster_3pt',
]).agg({
     'FG3A_Open_diff': 'mean',
    'FGA_Open_diff': 'mean', 
    'FG3A_Wide_Open_diff': 'mean',
    'FGA_Wide_Open_diff': 'mean',
    'FG3A_Tight_diff': 'mean',
    'FGA_Tight_diff': 'mean', 
    'FG3A_Very_Tight_diff': 'mean',
    'FGA_Very_Tight_diff': 'mean',
}).reset_index()



In [763]:
grouped_def

,OPPONENT_ID,SEASON_YEAR,OPPONENT_NAME,Cluster_3pt,FG3A_Open_diff,FGA_Open_diff,FG3A_Wide_Open_diff,FGA_Wide_Open_diff,FG3A_Tight_diff,FGA_Tight_diff,FG3A_Very_Tight_diff,FGA_Very_Tight_diff
0,1610612737,2023-24,Atlanta Hawks,0.0,-0.072858,-0.100825,0.271014,0.284278,0.009420,-0.451273,-0.043182,-0.315667
1,1610612737,2023-24,Atlanta Hawks,1.0,0.419937,0.635585,0.148277,0.167643,-0.017535,0.162537,-0.005207,-0.128738
2,1610612737,2023-24,Atlanta Hawks,2.0,0.126902,0.514978,0.037812,-0.005292,-0.030905,0.242959,-0.025785,-0.248158
3,1610612737,2023-24,Atlanta Hawks,3.0,0.536280,1.082425,0.485599,0.301257,-0.128153,-0.195048,-0.031649,-0.485313
4,1610612737,2023-24,Atlanta Hawks,4.0,0.077649,0.343295,0.031420,0.123430,-0.006217,0.380566,-0.010570,-0.340888
...,...,...,...,...,...,...,...,...,...,...,...,...
340,1610612766,2024-25,Charlotte Hornets,1.0,-0.128708,-0.350926,-0.225794,-0.273578,0.008325,-0.393374,-0.017504,-0.076062
341,1610612766,2024-25,Charlotte Hornets,2.0,0.122255,-0.085007,0.181894,0.165120,0.058945,-0.511488,-0.021500,-0.045936
342,1610612766,2024-25,Charlotte Hornets,3.0,0.111239,-0.328751,-0.080990,-0.172413,0.043480,-0.218755,0.003960,0.064128
343,1610612766,2024-25,Charlotte Hornets,4.0,0.010993,-0.069096,0.875360,0.854480,-0.059569,-0.120021,-0.003746,-0.201384


In [764]:
gamelogs_pu_team.columns

Index(['PLAYER_ID', 'PLAYER_NAME', 'GAME_DATE', 'OPPONENT_ID', 'OPPONENT_NAME',
       'Cluster_Pts', 'Cluster_3pt', 'GAMES_IN_WINDOW_60G', 'TEAM_ID',
       'TEAM_ABBREVIATION', 'GP', 'W', 'L', 'MIN', 'PULL_UP_FGM',
       'PULL_UP_FGA', 'PULL_UP_FG_PCT', 'PULL_UP_PTS', 'PULL_UP_FG3M',
       'PULL_UP_FG3A', 'PULL_UP_FG3_PCT', 'PULL_UP_EFG_PCT', 'Tracking', 'id',
       'SEASON_YEAR', 'GAMES_IN_WINDOW_TEAM_60G', 'PULL_UP_FGM_60G_Sum',
       'PULL_UP_FGA_60G_Sum', 'PULL_UP_FG3M_60G_Sum', 'PULL_UP_FG3A_60G_Sum'],
      dtype='object')

In [765]:
drives_team = gamelogs_drives_team.groupby(["SEASON_YEAR",'OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['MIN','DRIVES_60G_Sum', 'DRIVE_FGA_60G_Sum', 'DRIVE_PASSES_60G_Sum',
       'DRIVE_AST_60G_Sum', 'DRIVE_PTS_60G_Sum']].sum().reset_index()

In [766]:
drives_team['DRIVES/MIN_60Day'] = (drives_team['DRIVES_60G_Sum']/drives_team['MIN'] ).round(2)
drives_team['DRIVES_AST_PASS_RATE'] = (drives_team['DRIVE_AST_60G_Sum']/drives_team['DRIVE_PASSES_60G_Sum'] ).round(2)
drives_team['DRIVES_PASS_RATE'] = (drives_team['DRIVE_PASSES_60G_Sum']/drives_team['DRIVES_60G_Sum'] ).round(2)
drives_team['SHOTS_DRIVE_RATE'] =( drives_team['DRIVE_FGA_60G_Sum']/drives_team['DRIVES_60G_Sum'] ).round(2)
drives_team['PTS_PER_DRIVE'] =( drives_team['DRIVE_PTS_60G_Sum']/drives_team['DRIVES_60G_Sum'] ).round(2)

drives_60Day_sum = drives_team[['OPPONENT_ID','OPPONENT_NAME','GAME_DATE','DRIVES_60G_Sum','DRIVE_FGA_60G_Sum','DRIVE_PASSES_60G_Sum','DRIVE_AST_60G_Sum',
                                 'DRIVES/MIN_60Day','PTS_PER_DRIVE','DRIVES_AST_PASS_RATE','DRIVES_PASS_RATE','SHOTS_DRIVE_RATE']]

most_recent_mask = drives_60Day_sum['GAME_DATE'] == drives_60Day_sum.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
drives_60Day_new = drives_60Day_sum[most_recent_mask]

In [767]:
cs_team = gamelogs_cs_team.groupby(["SEASON_YEAR",'OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['MIN', 'CATCH_SHOOT_FGM_60G_Sum',
       'CATCH_SHOOT_FGA_60G_Sum', 'CATCH_SHOOT_FG3M_60G_Sum',
       'CATCH_SHOOT_FG3A_60G_Sum']].sum().reset_index()

In [768]:
cs_team['CS_FG2_PCT_60Day'] =((cs_team['CATCH_SHOOT_FGM_60G_Sum']-cs_team['CATCH_SHOOT_FG3M_60G_Sum'])/ (cs_team['CATCH_SHOOT_FGA_60G_Sum']-cs_team['CATCH_SHOOT_FG3A_60G_Sum'])).round(3)
cs_team['CS_FG3_PCT_60Day'] = ((cs_team['CATCH_SHOOT_FG3M_60G_Sum'])/ (cs_team['CATCH_SHOOT_FG3A_60G_Sum'])).round(3)
cs_team['CS_FGA_rate_60Day'] = ((cs_team['CATCH_SHOOT_FGA_60G_Sum'])/ (cs_team['MIN'])).round(2)
cs_team['CS_FG3A_rate_60Day'] = ((cs_team['CATCH_SHOOT_FG3A_60G_Sum'])/ (cs_team['MIN'])).round(2)

CS_60Day_sum = cs_team[['OPPONENT_ID','OPPONENT_NAME','GAME_DATE','MIN','CATCH_SHOOT_FGM_60G_Sum','CATCH_SHOOT_FGA_60G_Sum','CATCH_SHOOT_FG3M_60G_Sum','CATCH_SHOOT_FG3A_60G_Sum',
          'CS_FG2_PCT_60Day','CS_FG3_PCT_60Day','CS_FGA_rate_60Day','CS_FG3A_rate_60Day']]

most_recent_mask = CS_60Day_sum['GAME_DATE'] == CS_60Day_sum.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
CS_60Day_new = CS_60Day_sum[most_recent_mask]

In [769]:
pu_team = gamelogs_pu_team.groupby(["SEASON_YEAR",'OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['MIN', 'PULL_UP_FGM_60G_Sum',
       'PULL_UP_FGA_60G_Sum', 'PULL_UP_FG3M_60G_Sum', 'PULL_UP_FG3A_60G_Sum']].sum().reset_index()

In [770]:
pu_team['PU_FG2_PCT_60Day'] =((pu_team['PULL_UP_FGM_60G_Sum']-pu_team['PULL_UP_FG3M_60G_Sum'])/ (pu_team['PULL_UP_FGA_60G_Sum']-pu_team['PULL_UP_FG3A_60G_Sum'])).round(3)
pu_team['PU_FG3_PCT_60Day'] = ((pu_team['PULL_UP_FG3M_60G_Sum'])/ (pu_team['PULL_UP_FG3A_60G_Sum'])).round(3)
pu_team['PU_FGA_PCT_60Day'] = ((pu_team['PULL_UP_FGM_60G_Sum'])/ (pu_team['PULL_UP_FGA_60G_Sum'])).round(3)
pu_team['PU_FGA_rate_60Day'] = ((pu_team['PULL_UP_FGA_60G_Sum'])/ (pu_team['MIN'])).round(2)
pu_team['PU_FG3A_rate_60Day'] = ((pu_team['PULL_UP_FG3A_60G_Sum'])/ (pu_team['MIN'])).round(2)



PU_60Day_sum = pu_team[['OPPONENT_ID','OPPONENT_NAME','GAME_DATE','MIN','PULL_UP_FGM_60G_Sum','PULL_UP_FGA_60G_Sum','PULL_UP_FG3M_60G_Sum','PULL_UP_FG3A_60G_Sum',
          'PU_FG2_PCT_60Day','PU_FG3_PCT_60Day','PU_FGA_rate_60Day','PU_FG3A_rate_60Day','PU_FGA_PCT_60Day']]

most_recent_mask = PU_60Day_sum['GAME_DATE'] == PU_60Day_sum.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
PU_60Day_new = PU_60Day_sum[most_recent_mask]

In [771]:
#gamelogs_shotdetail_def.columns

In [772]:
gamelogs_shotdetail_def = matchups.merge(gamelogs_shotdetail, how='left')

In [773]:
#gamelogs_shotdetail
shotdetail_team = gamelogs_shotdetail_def.groupby(["SEASON_YEAR",'OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['FGA_3_AB', 'FGA_3_LC', 'FGA_3_RC', 'FGA_BC', 'FGA_Mid', 'FGA_Paint',
       'FGA_RA', 'FGM_3_AB', 'FGM_3_LC', 'FGM_3_RC', 'FGM_BC', 'FGM_Mid',
       'FGM_Paint', 'FGM_RA', 'FGA_3_AB_60G_Sum', 'FGA_3_LC_60G_Sum',
       'FGA_3_RC_60G_Sum', 'FGA_BC_60G_Sum', 'FGA_Mid_60G_Sum',
       'FGA_Paint_60G_Sum', 'FGA_RA_60G_Sum', 'FGM_3_AB_60G_Sum',
       'FGM_3_LC_60G_Sum', 'FGM_3_RC_60G_Sum', 'FGM_BC_60G_Sum',
       'FGM_Mid_60G_Sum', 'FGM_Paint_60G_Sum', 'FGM_RA_60G_Sum']].sum().reset_index()

In [774]:
most_recent_mask = shotdetail_team['GAME_DATE'] == shotdetail_team.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
shotdetail_60Day_new = shotdetail_team[most_recent_mask]

In [775]:
gamelogs_team = pd.DataFrame(gamelogs_with_rolling_w_opp.groupby(['SEASON_YEAR','OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['MIN','FGA','FG3A']].sum().reset_index())


gamelogs_team['GAME_DATE'] = pd.to_datetime(gamelogs_team['GAME_DATE'])
most_recent_mask = gamelogs_team['GAME_DATE'] == gamelogs_team.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
gamelogs_60Day_new = gamelogs_team[most_recent_mask]

In [776]:
#gamelogs_60Day_new

In [777]:
drives_add = drives_60Day_new[['OPPONENT_ID', 'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE',
       'DRIVES_PASS_RATE', 'SHOTS_DRIVE_RATE','PTS_PER_DRIVE']]
CS_add = CS_60Day_new[['OPPONENT_ID','CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day', 'CS_FGA_rate_60Day',
       'CS_FG3A_rate_60Day']]
PU_add = PU_60Day_new[['OPPONENT_ID','PU_FGA_PCT_60Day','PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day',
       'PU_FGA_rate_60Day', 'PU_FG3A_rate_60Day']]
shotdetail_add = shotdetail_60Day_new[['OPPONENT_ID', 'FGA_3_AB_60G_Sum', 'FGA_3_LC_60G_Sum',
       'FGA_3_RC_60G_Sum', 'FGA_BC_60G_Sum', 'FGA_Mid_60G_Sum',
       'FGA_Paint_60G_Sum', 'FGA_RA_60G_Sum', 'FGM_3_AB_60G_Sum',
       'FGM_3_LC_60G_Sum', 'FGM_3_RC_60G_Sum', 'FGM_BC_60G_Sum',
       'FGM_Mid_60G_Sum', 'FGM_Paint_60G_Sum', 'FGM_RA_60G_Sum']]
gamelog_add = gamelogs_60Day_new[['OPPONENT_ID', 'MIN','FGA','FG3A']]

"""passing_add = passing_60Day_new[['PLAYER_ID', 'PLAYER_NAME',  'PASSES_MIN_60G_Sum' ,'AST_60G_Sum',
       'AST_PASS_60G_Sum']]

rebounding_add = rebounding_60Day_new[['PLAYER_ID','OREB_PER_CHANCE', 'CONTEST_OREB_PER_CHANCE', 'DREB_PER_CHANCE',
       'CONTEST_DREB_PER_CHANCE']]
speeddistance_add = speeddistance_60Day_new[['PLAYER_ID','AVG_SPEED_OFF_60G_Sum', 'AVG_SPEED_DEF_60G_Sum', 'OFFmiles_PER_MIN',
       'DEFmiles_PER_MIN']]"""

"passing_add = passing_60Day_new[['PLAYER_ID', 'PLAYER_NAME',  'PASSES_MIN_60G_Sum' ,'AST_60G_Sum',\n       'AST_PASS_60G_Sum']]\n\nrebounding_add = rebounding_60Day_new[['PLAYER_ID','OREB_PER_CHANCE', 'CONTEST_OREB_PER_CHANCE', 'DREB_PER_CHANCE',\n       'CONTEST_DREB_PER_CHANCE']]\nspeeddistance_add = speeddistance_60Day_new[['PLAYER_ID','AVG_SPEED_OFF_60G_Sum', 'AVG_SPEED_DEF_60G_Sum', 'OFFmiles_PER_MIN',\n       'DEFmiles_PER_MIN']]"

In [778]:
#gamelog_add

In [779]:
#opponent_ranking_pts

In [780]:
teamtype_def_data['as_of'] = pd.to_datetime(teamtype_def_data['as_of'])
teamtype_def_data = pd.DataFrame(teamtype_def_data.sort_values(by=['TEAM_ID','PLAY_TYPE','as_of']).drop_duplicates(subset=['TEAM_ID','PLAY_TYPE'], keep='last'))

In [781]:

def analyze_defensive_playtypes(data):
    """
    Analyze defensive performance across different play types.
    Returns a DataFrame with all defensive metrics where higher scores = better defense.
    """
    if isinstance(data, list):
        df = pd.DataFrame(data)
    else:
        df = data.copy()
    
    def normalize_series(s):
        """Normalize a series to 0-100 scale"""
        min_val = s.values.min()
        max_val = s.values.max()
        if max_val == min_val:
            return pd.Series(50, index=s.index)
        return ((s - min_val) / (max_val - min_val) * 100).round(1)
    
    results = []
    for play_type, play_group in df.groupby('PLAY_TYPE'):
        # Calculate z-scores for key metrics
        metrics = ['TOV_POSS_PCT', 'SF_POSS_PCT', 'PPP', 'FG_PCT', 'SCORE_POSS_PCT', 'PLUSONE_POSS_PCT']
        z_scores = {}
        
        for metric in metrics:
            z_score = (play_group[metric] - play_group[metric].mean()) / play_group[metric].std()
            if metric == 'TOV_POSS_PCT':
                z_score *= -1  # Flip so higher = better defense
            z_scores[metric] = z_score
        
        disruption = (
            z_scores['TOV_POSS_PCT'] * 0.35 + 
            (z_scores['SF_POSS_PCT']) * 0.25 + 
            (z_scores['PPP']) * 0.40
        )
        
        defensive_efficiency = (
            (z_scores['FG_PCT']) * 0.40 +
            (z_scores['SCORE_POSS_PCT']) * 0.35 +
            (z_scores['PLUSONE_POSS_PCT']) * 0.25
        )
        
        overall = (disruption + defensive_efficiency) / 2
        
        for team in play_group['TEAM_NAME'].unique():
            team_mask = play_group['TEAM_NAME'] == team
            results.append({
                'TEAM_NAME': team,
                'PLAY_TYPE': play_type,
                'TEAM_ID': play_group.loc[team_mask, 'TEAM_ID'].iloc[0],
                'DISRUPTION_SCORE': disruption[team_mask].iloc[0],
                'DEF_EFFICIENCY_SCORE': defensive_efficiency[team_mask].iloc[0],
                'OVERALL_DEF_SCORE': overall[team_mask].iloc[0],
                'POSS_PCT': play_group.loc[team_mask, 'POSS_PCT'].iloc[0],
                'PPP_ALLOWED': play_group.loc[team_mask, 'PPP'].iloc[0],
                'OPP_FG_PCT': play_group.loc[team_mask, 'FG_PCT'].iloc[0],
                'TOV_FORCED_PCT': play_group.loc[team_mask, 'TOV_POSS_PCT'].iloc[0]
            })
    
    results_df = pd.DataFrame(results)
    
    # Normalize scores to 0-100 scale
    score_columns = ['DISRUPTION_SCORE', 'DEF_EFFICIENCY_SCORE', 'OVERALL_DEF_SCORE']
    for col in score_columns:
        results_df[col] = normalize_series(results_df[col])
    
    # Format percentages
    pct_columns = ['POSS_PCT', 'OPP_FG_PCT', 'TOV_FORCED_PCT']
    for col in pct_columns:
        results_df[col] = (results_df[col] * 100).round(1)
    
    # Round PPP to 2 decimals
    results_df['PPP_ALLOWED'] = results_df['PPP_ALLOWED'].round(3)
    
    # Add percentile rankings for key metrics
    for col in score_columns:
        results_df[f'{col}_PERCENTILE'] = (1 - results_df.groupby('PLAY_TYPE')[col].rank(pct=True)) * 100
    
    # Sort by overall defense score
    results_df = results_df.sort_values(['PLAY_TYPE', 'OVERALL_DEF_SCORE'], ascending=[True, False])
    
    # Reorder columns for clarity
    column_order = [
        'TEAM_ID', 'TEAM_NAME', 'PLAY_TYPE',
        'OVERALL_DEF_SCORE', 'OVERALL_DEF_SCORE_PERCENTILE',
        'DISRUPTION_SCORE', 'DISRUPTION_SCORE_PERCENTILE',
        'DEF_EFFICIENCY_SCORE', 'DEF_EFFICIENCY_SCORE_PERCENTILE',
        'PPP_ALLOWED', 'OPP_FG_PCT', 'TOV_FORCED_PCT', 'POSS_PCT'
    ]
    
    return results_df[column_order]

def get_team_defensive_profile(df, team_name):
    """
    Get a DataFrame of defensive metrics for a specific team across all play types.
    """
    results = analyze_defensive_playtypes(df)
    team_profile = results[results['TEAM_NAME'] == team_name].copy()
    
    # Add strength/weakness labels based on percentiles
    def get_performance_label(row):
        if row['OVERALL_DEF_SCORE_PERCENTILE'] >= 75:
            return 'Strength'
        elif row['OVERALL_DEF_SCORE_PERCENTILE'] <= 25:
            return 'Weakness'
        return 'Average'
    
    team_profile['PERFORMANCE_CATEGORY'] = team_profile.apply(get_performance_label, axis=1)
    
    return team_profile.sort_values('OVERALL_DEF_SCORE', ascending=False)

def get_play_type_insights(df, team_name):
    """
    Generate insights about a team's defensive performance by play type.
    All metrics are oriented so that higher scores indicate better defense.
    """
    play_type_details = analyze_defensive_playtypes(df)
    
    team_data = play_type_details[play_type_details['TEAM_NAME'] == team_name].copy()
    
    team_metrics = team_data.groupby('PLAY_TYPE').agg({
        'POSS_PCT': 'first',
        'PPP_ALLOWED': 'first',
        'OPP_FG_PCT': 'first',
        'TOV_FORCED_PCT': 'first',
        'DISRUPTION_SCORE': 'first',
        'DEF_EFFICIENCY_SCORE': 'first',
        'OVERALL_DEF_SCORE': 'first'
    }).round(2)
    
    insights = {
        'team': team_name,
        'metrics': team_metrics,
        'defensive_strengths': {
            'overall': team_metrics.nlargest(3, 'OVERALL_DEF_SCORE'),
            'disruption': team_metrics.nlargest(3, 'DISRUPTION_SCORE'),
            'efficiency': team_metrics.nlargest(3, 'DEF_EFFICIENCY_SCORE')
        },
        'defensive_weaknesses': {
            'overall': team_metrics.nsmallest(3, 'OVERALL_DEF_SCORE'),
            'disruption': team_metrics.nsmallest(3, 'DISRUPTION_SCORE'),
            'efficiency': team_metrics.nsmallest(3, 'DEF_EFFICIENCY_SCORE')
        },
        'most_frequent': team_metrics.nlargest(3, 'POSS_PCT')
    }
    
    return insights

def get_team_defensive_summary(df, team_name):
    """
    Get a formatted DataFrame summary of a team's defensive performance.
    """
    defensive_analysis = analyze_defensive_playtypes(df)
    team_data = defensive_analysis[defensive_analysis['TEAM_NAME'] == team_name].copy()
    
    # Sort by overall defensive score
    summary = team_data.sort_values('OVERALL_DEF_SCORE', ascending=False)
    
    # Add strength/weakness labels
    def get_strength_label(percentile):
        if percentile >= 75:
            return 'Strong'
        elif percentile <= 25:
            return 'Weak'
        return 'Average'
    
    summary['DEFENSIVE_STRENGTH'] = summary['OVERALL_DEF_SCORE_PERCENTILE'].apply(get_strength_label)
    
    return summary

def pivot_team_analysis(df):
    """
    Pivot player analysis to have play types as column suffixes.
    """
    # First get base analysis
    results = analyze_defensive_playtypes(df)
    
    # Select columns to pivot
    cols_to_pivot = [
        'OVERALL_DEF_SCORE','OVERALL_DEF_SCORE_PERCENTILE', 'DEF_EFFICIENCY_SCORE','DEF_EFFICIENCY_SCORE_PERCENTILE', 'DISRUPTION_SCORE','DISRUPTION_SCORE_PERCENTILE',
    ]
    
    # Create pivot tables for each metric and suffix with play type
    pivot_dfs = []
    
    # Keep player info for joining
    player_info = results[['TEAM_ID', 'TEAM_NAME']].drop_duplicates()
    
    for col in cols_to_pivot:
        pivot = pd.pivot_table(
            results,
            values=col,
            index=['TEAM_ID', 'TEAM_NAME',],
            columns='PLAY_TYPE',
            aggfunc='first'
        )
        
        # Rename columns to add metric as prefix
        pivot.columns = [f'{col}_{playtype}' for playtype in pivot.columns]
        pivot_dfs.append(pivot)
    
    # Combine all pivoted dataframes
    final_df = pd.concat(pivot_dfs, axis=1)
    final_df = final_df.reset_index()
    
    # Round numeric columns
    numeric_cols = final_df.select_dtypes(include=['float64']).columns
    final_df[numeric_cols] = final_df[numeric_cols].round(1)
    
    return final_df

In [782]:
results_playtype = pivot_team_analysis(teamtype_def_data)

In [783]:
results_playtype.rename(columns={'TEAM_ID':'OPPONENT_ID','TEAM_NAME':'OPPONENT_NAME'},inplace=True )

In [784]:
results_playtype.sort_values(by='OVERALL_DEF_SCORE_OffScreen')

,OPPONENT_ID,OPPONENT_NAME,OVERALL_DEF_SCORE_Cut,OVERALL_DEF_SCORE_Handoff,OVERALL_DEF_SCORE_Isolation,OVERALL_DEF_SCORE_Misc,OVERALL_DEF_SCORE_OffRebound,OVERALL_DEF_SCORE_OffScreen,OVERALL_DEF_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRRollMan,OVERALL_DEF_SCORE_Postup,OVERALL_DEF_SCORE_Spotup,OVERALL_DEF_SCORE_Transition,OVERALL_DEF_SCORE_PERCENTILE_Cut,OVERALL_DEF_SCORE_PERCENTILE_Handoff,OVERALL_DEF_SCORE_PERCENTILE_Isolation,OVERALL_DEF_SCORE_PERCENTILE_Misc,OVERALL_DEF_SCORE_PERCENTILE_OffRebound,OVERALL_DEF_SCORE_PERCENTILE_OffScreen,OVERALL_DEF_SCORE_PERCENTILE_PRBallHandler,OVERALL_DEF_SCORE_PERCENTILE_PRRollMan,OVERALL_DEF_SCORE_PERCENTILE_Postup,OVERALL_DEF_SCORE_PERCENTILE_Spotup,OVERALL_DEF_SCORE_PERCENTILE_Transition,DEF_EFFICIENCY_SCORE_Cut,DEF_EFFICIENCY_SCORE_Handoff,DEF_EFFICIENCY_SCORE_Isolation,DEF_EFFICIENCY_SCORE_Misc,DEF_EFFICIENCY_SCORE_OffRebound,DEF_EFFICIENCY_SCORE_OffScreen,DEF_EFFICIENCY_SCORE_PRBallHandler,DEF_EFFICIENCY_SCORE_PRRollMan,DEF_EFFICIENCY_SCORE_Postup,DEF_EFFICIENCY_SCORE_Spotup,DEF_EFFICIENCY_SCORE_Transition,DEF_EFFICIENCY_SCORE_PERCENTILE_Cut,DEF_EFFICIENCY_SCORE_PERCENTILE_Handoff,DEF_EFFICIENCY_SCORE_PERCENTILE_Isolation,DEF_EFFICIENCY_SCORE_PERCENTILE_Misc,DEF_EFFICIENCY_SCORE_PERCENTILE_OffRebound,DEF_EFFICIENCY_SCORE_PERCENTILE_OffScreen,DEF_EFFICIENCY_SCORE_PERCENTILE_PRBallHandler,DEF_EFFICIENCY_SCORE_PERCENTILE_PRRollMan,DEF_EFFICIENCY_SCORE_PERCENTILE_Postup,DEF_EFFICIENCY_SCORE_PERCENTILE_Spotup,DEF_EFFICIENCY_SCORE_PERCENTILE_Transition,DISRUPTION_SCORE_Cut,DISRUPTION_SCORE_Handoff,DISRUPTION_SCORE_Isolation,DISRUPTION_SCORE_Misc,DISRUPTION_SCORE_OffRebound,DISRUPTION_SCORE_OffScreen,DISRUPTION_SCORE_PRBallHandler,DISRUPTION_SCORE_PRRollMan,DISRUPTION_SCORE_Postup,DISRUPTION_SCORE_Spotup,DISRUPTION_SCORE_Transition,DISRUPTION_SCORE_PERCENTILE_Cut,DISRUPTION_SCORE_PERCENTILE_Handoff,DISRUPTION_SCORE_PERCENTILE_Isolation,DISRUPTION_SCORE_PERCENTILE_Misc,DISRUPTION_SCORE_PERCENTILE_OffRebound,DISRUPTION_SCORE_PERCENTILE_OffScreen,DISRUPTION_SCORE_PERCENTILE_PRBallHandler,DISRUPTION_SCORE_PERCENTILE_PRRollMan,DISRUPTION_SCORE_PERCENTILE_Postup,DISRUPTION_SCORE_PERCENTILE_Spotup,DISRUPTION_SCORE_PERCENTILE_Transition
7,1610612744,Golden State Warriors,44.7,34.5,48.0,55.9,63.7,15.2,41.2,37.6,69.6,58.1,24.2,63.3,83.3,40.0,33.3,20.0,96.7,76.7,78.3,10.0,30.0,93.3,51.9,42.9,46.8,50.8,63.7,21.3,52.2,50.4,69.1,58.3,19.1,41.7,60.0,46.7,46.7,20.0,96.7,45.0,41.7,16.7,23.3,90.0,36.5,25.9,48.4,59.7,61.5,10.7,29.2,23.9,67.4,56.2,30.6,76.7,96.7,40.0,26.7,26.7,96.7,86.7,93.3,13.3,30.0,86.7
10,1610612747,Los Angeles Lakers,57.3,74.4,37.8,76.8,53.8,21.0,5.8,55.7,64.0,45.2,28.7,30.0,3.3,73.3,3.3,46.7,93.3,96.7,36.7,23.3,70.0,83.3,59.2,81.7,39.5,69.9,51.7,25.5,0.0,53.7,63.8,45.7,26.3,30.0,3.3,63.3,10.0,50.0,83.3,96.7,33.3,20.0,53.3,86.7,53.6,63.6,36.1,80.9,54.6,17.7,14.7,56.3,62.1,44.1,32.0,40.0,13.3,86.7,3.3,43.3,93.3,96.7,33.3,20.0,66.7,83.3
6,1610612743,Denver Nuggets,41.5,44.3,42.1,66.1,31.8,25.4,21.9,54.2,35.5,87.5,53.5,66.7,50.0,60.0,16.7,80.0,90.0,90.0,46.7,73.3,0.0,46.7,41.4,37.6,36.2,58.9,29.1,21.9,22.0,60.4,32.2,100.0,43.0,66.7,63.3,70.0,26.7,80.0,93.3,90.0,20.0,83.3,0.0,63.3,41.3,51.0,48.0,71.3,35.2,30.2,23.2,46.2,39.2,70.0,63.4,63.3,43.3,43.3,13.3,76.7,80.0,90.0,63.3,68.3,13.3,16.7
27,1610612764,Washington Wizards,71.1,38.6,82.9,71.0,51.1,26.5,50.8,77.8,43.3,53.9,74.1,6.7,66.7,3.3,13.3,50.0,86.7,53.3,3.3,60.0,33.3,3.3,69.7,44.6,76.1,65.7,50.1,26.1,39.1,77.1,37.0,51.8,70.2,13.3,55.0,6.7,16.7,56.7,80.0,73.3,6.7,73.3,36.7,11.7,69.7,32.2,86.2,73.8,51.1,27.9,62.1,75.1,49.5,54.7,75.2,10.0,90.0,0.0,10.0,56.7,86.7,20.0,0.0,50.0,36.7,6.7
22,1610612759,San Antonio Spurs,9.2,43.9,44.3,25.0,62.8,30.7,57.4,58.8,59.8,45.3,53.3,96.7,53.3,53.3,93.3,23.3,83.3,33.3,26.7,30.0,66.7,53.3,5.2,48.4,46.7,22.1,61.3,23.5,56.6,62.6,61.0,47.8,48.5,96.7,38.3,50.0,93.3,30.0,90.0,26.7,16.7,26.7,40.0,53.3,15.9,38.6,41.2,29.1,62.2,38.9,56.6,53.0,56.8,42.0,57.2,96.7,70.0,66.7,86.7,23.3,73.3,33.3,53.3,33.3,70.0,33.3
14,

In [785]:
#playtype_def_data
#playtype_def_df = pd.DataFrame(teamtype_def_data[['TEAM_ID','TEAM_NAME','GP','PLAY_TYPE','PTS','POSS_PCT','POSS','PPP','FGA','FGM','FG_PCT']])

In [786]:
#playtype_def_df['SHOT_FREQ'] = (playtype_def_df['FGA']/playtype_def_df['POSS']).round(2)
#playtype_def_df['POSS/G'] = (playtype_def_df['POSS']/playtype_def_df['GP']).round(1)
#playtype_def_df['FGA/G'] = (playtype_def_df['FGA']/playtype_def_df['GP']).round(1)
#playtype_def_df['PTS/MAKE'] = (playtype_def_df['PTS']/playtype_def_df['FGM']).round(2)

In [787]:
#playtype_def_pivot = playtype_def_df.pivot_table(index=[ 'TEAM_ID','TEAM_NAME' ], 
#                          columns='PLAY_TYPE', 
#                          values=['POSS_PCT', 'POSS/G', 'PPP', 'FGA/G','SHOT_FREQ','FG_PCT','PTS/MAKE'])

# Flatten the multi-index columns and rename them with the format `metric_PLAYTYPE`
#playtype_def_pivot.columns = [f'{metric}_{play_type}' for metric, play_type in playtype_def_pivot.columns]

# Reset index to turn multi-index back into regular columns
#playtype_def_pivot.reset_index(inplace=True)

In [788]:
#playtype_def_pivot.rename(columns={'TEAM_ID':'OPPONENT_ID','TEAM_NAME':'OPPONENT_NAME'}, inplace=True)

In [789]:
PU_add

,OPPONENT_ID,PU_FGA_PCT_60Day,PU_FG2_PCT_60Day,PU_FG3_PCT_60Day,PU_FGA_rate_60Day,PU_FG3A_rate_60Day
2522,1610612737,0.441,0.539,0.318,5.41,2.40
2588,1610612738,0.404,0.471,0.327,4.80,2.25
2652,1610612739,0.334,0.443,0.216,5.06,2.42
2717,1610612740,0.385,0.536,0.236,4.75,2.39
2781,1610612741,0.428,0.476,0.381,3.78,1.88
2846,1610612742,0.507,0.557,0.425,4.30,1.64
2909,1610612743,0.383,0.409,0.345,8.37,3.41
2972,1610612744,0.396,0.454,0.323,5.08,2.24
3036,1610612745,0.345,0.370,0.301,4.56,1.66
3101,1610612746,0.385,0.450,0.315,7.16,3.42


In [790]:
# List of DataFrames to merge
dfs_to_merge = [
    gamelog_add,PU_add, CS_add, drives_add, shotdetail_add
]

# Loop through each DataFrame in the list and merge, dropping 'PLAYER_NAME' and 'GAME_DATE' columns
for df in dfs_to_merge:
    results_playtype = results_playtype.merge(df, how='left')

# The final merged DataFrame
df_playtypes = results_playtype

In [791]:
df_playtypes.columns

Index(['OPPONENT_ID', 'OPPONENT_NAME', 'OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition',
       'OVERALL_DEF_SCORE_PERCENTILE_Cut',
       'OVERALL_DEF_SCORE_PERCENTILE_Handoff',
       'OVERALL_DEF_SCORE_PERCENTILE_Isolation',
       'OVERALL_DEF_SCORE_PERCENTILE_Misc',
       'OVERALL_DEF_SCORE_PERCENTILE_OffRebound',
       'OVERALL_DEF_SCORE_PERCENTILE_OffScreen',
       'OVERALL_DEF_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_DEF_SCORE_PERCENTILE_PRRollMan',
       'OVERALL_DEF_SCORE_PERCENTILE_Postup',
       'OVERALL_DEF_SCORE_PERCENTILE_Spotup',
       'OVERALL_DEF_SCORE_PERCENTILE_Transition', 'DEF_EFFICIENCY_SCORE_Cut',
       'DEF_EFFICIENCY_SCORE_Handoff', 'DEF_

In [792]:
df_scorer_cat = pd.DataFrame(df_playtypes[['OPPONENT_ID','OPPONENT_NAME','MIN','FGA','FG3A','OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition','DEF_EFFICIENCY_SCORE_Cut',
       'DEF_EFFICIENCY_SCORE_Handoff', 'DEF_EFFICIENCY_SCORE_Isolation',
       'DEF_EFFICIENCY_SCORE_Misc', 'DEF_EFFICIENCY_SCORE_OffRebound',
       'DEF_EFFICIENCY_SCORE_OffScreen', 'DEF_EFFICIENCY_SCORE_PRBallHandler',
       'DEF_EFFICIENCY_SCORE_PRRollMan', 'DEF_EFFICIENCY_SCORE_Postup',
       'DEF_EFFICIENCY_SCORE_Spotup', 'DEF_EFFICIENCY_SCORE_Transition','DISRUPTION_SCORE_Cut',
       'DISRUPTION_SCORE_Handoff', 'DISRUPTION_SCORE_Isolation',
       'DISRUPTION_SCORE_Misc', 'DISRUPTION_SCORE_OffRebound',
       'DISRUPTION_SCORE_OffScreen', 'DISRUPTION_SCORE_PRBallHandler',
       'DISRUPTION_SCORE_PRRollMan', 'DISRUPTION_SCORE_Postup',
       'DISRUPTION_SCORE_Spotup', 'DISRUPTION_SCORE_Transition',
       'PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day','PU_FGA_PCT_60Day', 'PU_FGA_rate_60Day',
       'PU_FG3A_rate_60Day', 'CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day',
       'CS_FGA_rate_60Day', 'CS_FG3A_rate_60Day','PTS_PER_DRIVE', 'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE',
       'DRIVES_PASS_RATE','SHOTS_DRIVE_RATE','FGA_3_AB_60G_Sum', 'FGA_3_LC_60G_Sum',
       'FGA_3_RC_60G_Sum', 'FGA_BC_60G_Sum', 'FGA_Mid_60G_Sum',
       'FGA_Paint_60G_Sum', 'FGA_RA_60G_Sum', 'FGM_3_AB_60G_Sum',
       'FGM_3_LC_60G_Sum', 'FGM_3_RC_60G_Sum', 'FGM_BC_60G_Sum',
       'FGM_Mid_60G_Sum', 'FGM_Paint_60G_Sum', 'FGM_RA_60G_Sum']])

In [793]:
df_scorer_cat.fillna(0, inplace=True)

In [794]:
df_scorer_cat['RA_PCT'] = df_scorer_cat['FGM_RA_60G_Sum']/df_scorer_cat['FGA_RA_60G_Sum']
df_scorer_cat['Paint_PCT'] = df_scorer_cat['FGM_Paint_60G_Sum']/df_scorer_cat['FGA_Paint_60G_Sum']
df_scorer_cat['RA_RATE'] = df_scorer_cat['FGA_RA_60G_Sum']/df_scorer_cat['MIN']
df_scorer_cat['Paint_RATE'] = df_scorer_cat['FGA_Paint_60G_Sum']/df_scorer_cat['MIN']

df_scorer_cat['MID_PCT'] = df_scorer_cat['FGM_Mid_60G_Sum']/df_scorer_cat['FGA_Mid_60G_Sum']
df_scorer_cat['FG3_PCT'] = (df_scorer_cat['FGM_3_RC_60G_Sum']+df_scorer_cat['FGM_3_LC_60G_Sum']+df_scorer_cat['FGM_3_AB_60G_Sum'])/df_scorer_cat['FG3A']
df_scorer_cat['MID_RATE'] = df_scorer_cat['FGA_Mid_60G_Sum']/df_scorer_cat['MIN']
df_scorer_cat['FG3_RATE'] = df_scorer_cat['FG3A']/df_scorer_cat['MIN']

In [795]:
# Drives
league_avg_pts_per_drive = df_scorer_cat['PTS_PER_DRIVE'].mean()
league_avg_DSR = df_scorer_cat['SHOTS_DRIVE_RATE'].mean()
league_avg_drives_min = df_scorer_cat['DRIVES/MIN_60Day'].mean()

#ISO
league_avg_iso_shot = df_scorer_cat['OVERALL_DEF_SCORE_Isolation'].mean()
#league_avg_iso_pct = df_scorer_cat['FG_PCT_Isolation'].mean()

#PNR BallHandler
league_avg_pnrh_shot = df_scorer_cat['OVERALL_DEF_SCORE_PRBallHandler'].mean()
#league_avg_pnrh_pct = df_scorer_cat['FG_PCT_PRBallHandler'].mean()

#PNRRollman
league_avg_pnrman_shot = df_scorer_cat['OVERALL_DEF_SCORE_PRRollMan'].mean()
#eague_avg_pnrman_pct = df_scorer_cat['FG_PCT_PRRollMan'].mean()

#PNRRollman
league_avg_putback_shot = df_scorer_cat['OVERALL_DEF_SCORE_OffRebound'].mean()
#league_avg_putback_pct = df_scorer_cat['FG_PCT_OffRebound'].mean()

#_Cut
league_avg_cut_shot = df_scorer_cat['OVERALL_DEF_SCORE_Cut'].mean()
#league_avg_cut_pct = df_scorer_cat['FG_PCT_Cut'].mean()

#_Postup
league_avg_postup_shot = df_scorer_cat['OVERALL_DEF_SCORE_Postup'].mean()
#league_avg_cut_pct = df_scorer_cat['FG_PCT_Postup'].mean()

#_Pullup
league_avg_fg_pct_PU = df_scorer_cat['PU_FGA_PCT_60Day'].mean()
league_avg_fga_per_min_PU = df_scorer_cat['PU_FGA_rate_60Day'].mean()

#Paint Shots
league_avg_ra_pct = df_scorer_cat['RA_PCT'].mean()
league_avg_paint_pct = df_scorer_cat['Paint_PCT'].mean()
league_avg_ra_min = df_scorer_cat['RA_RATE'].mean()
league_avg_paint_min = df_scorer_cat['Paint_RATE'].mean()
league_avg_mid_pct = df_scorer_cat['MID_PCT'].mean()
league_avg_fg3_pct = df_scorer_cat['FG3_PCT'].mean()
league_avg_mid_min = df_scorer_cat['MID_RATE'].mean()
league_avg_fg3_min = df_scorer_cat['FG3_RATE'].mean()



       

df_scorer_cat['pu_eff_ratio'] = (df_scorer_cat['PU_FGA_PCT_60Day'] / league_avg_fg_pct_PU) 
df_scorer_cat['drive_eff_ratio'] = df_scorer_cat['PTS_PER_DRIVE'] / league_avg_pts_per_drive
#df_scorer_cat['pnr_eff_ratio'] = df_scorer_cat['FG_PCT_PRBallHandler'] / league_avg_pnrh_pct
#df_scorer_cat['pnrman_eff_ratio'] = df_scorer_cat['FG_PCT_PRRollMan'] / league_avg_pnrman_pct
#df_scorer_cat['iso_eff_ratio'] = df_scorer_cat['FG_PCT_Isolation'] / league_avg_iso_pct
#df_scorer_cat['cut_eff_ratio'] = df_scorer_cat['FG_PCT_Cut'] / league_avg_cut_pct
#df_scorer_cat['putback_eff_ratio'] = df_scorer_cat['FG_PCT_OffRebound'] / league_avg_putback_pct
df_scorer_cat['ra_eff_ratio'] = df_scorer_cat['RA_PCT'] / league_avg_ra_pct
df_scorer_cat['paint_eff_ratio'] = df_scorer_cat['Paint_PCT'] / league_avg_paint_pct
df_scorer_cat['mid_eff_ratio'] = df_scorer_cat['MID_PCT'] / league_avg_mid_pct
df_scorer_cat['fg3_eff_ratio'] = df_scorer_cat['FG3_PCT'] / league_avg_fg3_pct
    
    # Calculate volume vs. league average ratios
df_scorer_cat['pu_vol_ratio'] = (df_scorer_cat['PU_FGA_rate_60Day'] / league_avg_fga_per_min_PU)
df_scorer_cat['drive_vol_ratio'] = df_scorer_cat['DRIVES/MIN_60Day'] / league_avg_drives_min
#df_scorer_cat['pnr_vol_ratio'] = df_scorer_cat['FGA/G_PRBallHandler'] / league_avg_pnrh_shot
#df_scorer_cat['pnrman_vol_ratio'] = df_scorer_cat['FGA/G_PRRollMan'] / league_avg_pnrman_shot
#df_scorer_cat['iso_vol_ratio'] = df_scorer_cat['FGA/G_Isolation'] / league_avg_iso_shot
#df_scorer_cat['cut_vol_ratio'] = df_scorer_cat['FGA/G_Cut'] / league_avg_cut_shot
#df_scorer_cat['putback_vol_ratio'] = df_scorer_cat['FGA/G_OffRebound'] / league_avg_putback_shot
df_scorer_cat['ra_vol_ratio'] = df_scorer_cat['RA_RATE'] / league_avg_ra_min
df_scorer_cat['paint_vol_ratio'] = df_scorer_cat['Paint_RATE'] / league_avg_paint_min
df_scorer_cat['mid_vol_ratio'] = df_scorer_cat['MID_RATE'] / league_avg_mid_min
df_scorer_cat['fg3_vol_ratio'] = df_scorer_cat['FG3_RATE'] / league_avg_fg3_min

#drives
df_scorer_cat['driving_score'] = (
    df_scorer_cat['drive_eff_ratio'] * 0.55 +
    df_scorer_cat['drive_vol_ratio'] * 0.45
)

#iso
df_scorer_cat['iso_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_Isolation'] / league_avg_iso_shot) # Scale by frequency to reward volume
    )
#putback
df_scorer_cat['putback_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_OffRebound'] / league_avg_putback_shot) # Scale by frequency to reward volume
    )

#cut
df_scorer_cat['cut_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_Cut'] / league_avg_cut_shot) # Scale by frequency to reward volume
    )

#postup
df_scorer_cat['postup_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_Postup'] / league_avg_postup_shot) # Scale by frequency to reward volume
    )
#pullup
df_scorer_cat['pullup_score'] = (
    df_scorer_cat['pu_eff_ratio'] * 0.55 +
    df_scorer_cat['pu_vol_ratio'] * 0.45
)

#pnrman_score
df_scorer_cat['pnrman_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_PRRollMan'] / league_avg_pnrman_shot) # Scale by frequency to reward volume
    )

    # PnR Score (30% weight)
df_scorer_cat['pnr_score'] = (
        (df_scorer_cat['OVERALL_DEF_SCORE_PRBallHandler'] / league_avg_pnrh_shot) # Scale by frequency to reward volume
    )

    # mid
df_scorer_cat['mid_score'] = (
    df_scorer_cat['mid_eff_ratio'] * 0.55 +
    df_scorer_cat['mid_vol_ratio'] * 0.45
)

    # fg3
df_scorer_cat['fg3_score'] = (
    df_scorer_cat['fg3_eff_ratio'] * 0.55 +
    df_scorer_cat['fg3_vol_ratio'] * 0.45
)

# Interior Score (30% weight)
df_scorer_cat['interior_score'] = (
    (df_scorer_cat['ra_eff_ratio'] * 0.55 + df_scorer_cat['ra_vol_ratio'] * 0.45) * 0.5 +  # RA component
    (df_scorer_cat['paint_eff_ratio'] * 0.55 + df_scorer_cat['paint_vol_ratio'] * 0.45) * 0.5  # Paint component
)

df_scorer_cat['penetrator_score'] = (
        df_scorer_cat['driving_score'] * 0.40 +    # Driving ability
        df_scorer_cat['pnr_score'] * 0.30 +        # Pick and Roll scoring
        df_scorer_cat['interior_score'] * 0.30  # Interior scoring
    ).round(4)

average_score = df_scorer_cat['penetrator_score'].mean()
df_scorer_cat['penetrator_score'] = (df_scorer_cat['penetrator_score'] / average_score) * 100



df_scorer_cat['rim_runner_score'] = (
        df_scorer_cat['pnrman_score'] * 0.25 +    
        df_scorer_cat['cut_score'] * 0.25 + 
        df_scorer_cat['putback_score'] * 0.20 + 
        df_scorer_cat['interior_score'] * 0.30     
    ).round(4)

average_score = df_scorer_cat['rim_runner_score'].mean()
df_scorer_cat['rim_runner_score'] = (df_scorer_cat['rim_runner_score'] / average_score) * 100

df_scorer_cat['pure_score'] = (
        df_scorer_cat['iso_score'] * 0.30 + 
        df_scorer_cat['pullup_score'] * 0.30 +  
        df_scorer_cat['mid_score'] * 0.15 +  # Driving ability
        df_scorer_cat['fg3_score'] * 0.20 +        # Pick and Roll scoring
        #df_scorer_cat['driving_score'] * 0.20 +
        df_scorer_cat['driving_score'] * 0.25 # Interior scoring
    ).round(4)



average_score = df_scorer_cat['pure_score'].mean()
df_scorer_cat['pure_score'] = (df_scorer_cat['pure_score'] / average_score) * 100

df_scorer_cat.fillna(0,inplace=True)


In [796]:
df_def_type_scores = pd.DataFrame(df_scorer_cat[['OPPONENT_ID','OPPONENT_NAME','penetrator_score','pure_score','rim_runner_score','OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition','pullup_score','mid_score','fg3_score','driving_score']])

In [797]:
df_def_type_scores

,OPPONENT_ID,OPPONENT_NAME,penetrator_score,pure_score,rim_runner_score,OVERALL_DEF_SCORE_Cut,OVERALL_DEF_SCORE_Handoff,OVERALL_DEF_SCORE_Isolation,OVERALL_DEF_SCORE_Misc,OVERALL_DEF_SCORE_OffRebound,OVERALL_DEF_SCORE_OffScreen,OVERALL_DEF_SCORE_PRBallHandler,OVERALL_DEF_SCORE_PRRollMan,OVERALL_DEF_SCORE_Postup,OVERALL_DEF_SCORE_Spotup,OVERALL_DEF_SCORE_Transition,pullup_score,mid_score,fg3_score,driving_score
0,1610612737,Atlanta Hawks,120.790403,111.533024,116.37,66.3,35.9,47.3,52.2,58.8,37.7,67.4,37.7,64.5,64.2,53.4,1.146173,1.339799,1.302388,0.991759
1,1610612738,Boston Celtics,94.080314,97.608062,82.74,40.8,67.7,66.3,35.5,70.8,36.7,58.7,24.4,40.8,14.2,17.7,1.035266,0.600381,0.692936,0.929242
2,1610612739,Cleveland Cavaliers,97.410325,108.858031,80.87,36.8,43.3,53.2,40.8,7.3,34.5,27.9,38.0,30.4,45.6,28.2,0.958993,1.335519,1.230181,1.005766
3,1610612740,New Orleans Pelicans,108.280361,106.066372,128.85,55.1,70.9,44.5,55.2,62.0,44.7,47.0,73.2,62.2,38.1,83.7,1.003176,1.208452,1.346720,1.011782
4,1610612741,Chicago Bulls,107.690359,112.183022,130.68,72.9,61.8,63.2,88.6,61.9,83.7,57.3,62.9,67.6,51.3,50.0,0.973502,1.142145,1.397517,0.889271
5,1610612742,Dallas Mavericks,120.340401,125.232985,93.11,33.6,43.4,64.7,56.8,30.0,46.3,53.7,48.3,48.7,49.5,26.0,1.136407,1.363911,1.333938,1.201480
6,1610612743,Denver Nuggets,77.270258,96.141400,76.53,41.5,44.3,42.1,66.1,31.8,25.4,21.9,54.2,35.5,87.5,53.5,1.342187,0.600952,0.522874,1.210153
7,1610612744,Golden State Warriors,88.030293,91.074747,85.73,44.7,34.5,48.0,55.9,63.7,15.2,41.2,37.6,69.6,58.1,24.2,1.050188,0.566272,0.629931,1.110152
8,1610612745,Houston Rockets,83.410278,91.649745,96.54,35.0,44.6,65.1,58.7,69.7,52.5,35.1,58.2,29.2,17.4,39.7,0.927615,0.451812,0.538806,1.013756
9,1610612746,LA Clippers,88.070294,98.533060,71.03,33.2,26.2,40.8,74.8,47.3,49.5,35.0,19.3,47.6,48.3,56.4,1.230789,0.887261,0.878636,1.033210


In [798]:
df_def_type_scores['as_of'] = pd.to_datetime(today)
df_def_type_scores['id'] = df_def_type_scores['as_of'].astype(str)+"_"+df_def_type_scores['OPPONENT_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "opponent_def_type_scores_new"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    df_def_type_scores.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = df_def_type_scores[~df_def_type_scores['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'opponent_def_type_scores_new' already exists. Checking for new records...
Inserted 30 new records into 'opponent_def_type_scores_new'.


In [799]:
closestdefender_matchup.loc[closestdefender_matchup['Cluster_3pt']==1]['OPPONENT_ID'].nunique()

30

In [800]:
closestdefender_diff_mean = closestdefender_matchup.groupby(['OPPONENT_ID'])[['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff']].mean().reset_index()

closestdefender_diff_mean_pts = closestdefender_matchup.groupby(['OPPONENT_ID','Cluster_Pts'])[['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff']].mean().reset_index()

closestdefender_diff_mean_3pt = closestdefender_matchup.groupby(['OPPONENT_ID','Cluster_3pt'])[['FG3A_Open_diff', 
       'FG3A_Wide_Open_diff',  'FG3A_Tight_diff','FG3A_Very_Tight_diff', ]].mean().reset_index()



In [801]:
closestdefender_diff_mean_3pt

,OPPONENT_ID,Cluster_3pt,FG3A_Open_diff,FG3A_Wide_Open_diff,FG3A_Tight_diff,FG3A_Very_Tight_diff
0,1610612737,0.0,-0.007144,0.369104,-0.052042,0.045592
1,1610612737,1.0,0.339608,0.248073,0.019623,0.003860
2,1610612737,2.0,0.184696,0.042969,-0.032134,-0.015510
3,1610612737,3.0,0.414649,0.558845,-0.074653,-0.019574
4,1610612737,4.0,0.149588,-0.208710,-0.018755,0.012213
...,...,...,...,...,...,...
175,1610612766,1.0,0.194049,0.026607,-0.007431,-0.011609
176,1610612766,2.0,0.124795,0.164754,-0.020822,-0.011483
177,1610612766,3.0,0.221026,-0.065791,0.076375,0.045610
178,1610612766,4.0,0.034994,0.419231,0.005226,-0.006299


In [802]:
closestdefender_diff_mean_3pt.loc[closestdefender_diff_mean_3pt['Cluster_3pt']==1]['OPPONENT_ID'].nunique()

30

In [803]:
closestdefender_diff_sum = closestdefender_matchup.groupby(['OPPONENT_ID','OPPONENT_NAME','GAME_DATE'])[['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff']].sum().reset_index()

closestdefender_diff_sum_pts = closestdefender_matchup.groupby(['OPPONENT_ID','OPPONENT_NAME','Cluster_Pts','GAME_DATE'])[['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff']].sum().reset_index()

closestdefender_diff_sum_3pt = closestdefender_matchup.groupby(['OPPONENT_ID','OPPONENT_NAME','Cluster_3pt','GAME_DATE'])[['FG3A_Open_diff', 
       'FG3A_Wide_Open_diff',  'FG3A_Tight_diff','FG3A_Very_Tight_diff', ]].sum().reset_index()

In [804]:
closestdefender_diff_sum_3pt

,OPPONENT_ID,OPPONENT_NAME,Cluster_3pt,GAME_DATE,FG3A_Open_diff,FG3A_Wide_Open_diff,FG3A_Tight_diff,FG3A_Very_Tight_diff
0,1610612737,Atlanta Hawks,0.0,2023-11-22,-0.923077,0.307692,-0.769231,-0.153846
1,1610612737,Atlanta Hawks,0.0,2023-11-30,0.444444,1.055556,-0.277778,0.000000
2,1610612737,Atlanta Hawks,0.0,2023-12-18,-0.769231,-1.846154,-0.076923,0.000000
3,1610612737,Atlanta Hawks,0.0,2024-01-15,-0.692308,2.512821,-0.153846,0.000000
4,1610612737,Atlanta Hawks,0.0,2024-02-14,-0.933333,1.066667,1.266667,0.000000
...,...,...,...,...,...,...,...,...
15964,1610612766,Charlotte Hornets,5.0,2025-02-27,-0.616667,-1.500000,1.150000,-0.016667
15965,1610612766,Charlotte Hornets,5.0,2025-03-03,-0.533333,4.100000,-1.133333,-0.150000
15966,1610612766,Charlotte Hornets,5.0,2025-03-05,-0.950000,0.066667,-0.950000,-0.100000
15967,1610612766,Charlotte Hornets,5.0,2025-03-07,-1.016667,-0.716667,-0.566667,-0.016667


In [805]:
gamelogs_def_closestdefender_avg = calculate_team_rolling_stats(
    gamelogs=closestdefender_diff_sum,
    rolling_columns=['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff'],
    method = 'average'
)

gamelogs_def_closestdefender_avg_pts = calculate_team_rolling_stats(
    gamelogs=closestdefender_diff_sum_pts,
    rolling_columns=['FG3A_Open_diff', 'FGA_Open_diff',
       'FG3A_Wide_Open_diff', 'FGA_Wide_Open_diff', 'FG3A_Tight_diff',
       'FGA_Tight_diff', 'FG3A_Very_Tight_diff', 'FGA_Very_Tight_diff'],
    method = 'average'
)

gamelogs_def_closestdefender_avg_3pt = calculate_team_rolling_stats(
    gamelogs=closestdefender_diff_sum_3pt,
    group_columns=['OPPONENT_ID','Cluster_3pt'],
    window_size = 30,
    rolling_columns=['FG3A_Open_diff', 'FG3A_Wide_Open_diff', 'FG3A_Tight_diff', 'FG3A_Very_Tight_diff'],
    method = 'average'
)

In [806]:
gamelogs_def_closestdefender_avg_3pt

,OPPONENT_ID,OPPONENT_NAME,Cluster_3pt,GAME_DATE,FG3A_Open_diff,FG3A_Wide_Open_diff,FG3A_Tight_diff,FG3A_Very_Tight_diff,GAMES_IN_WINDOW_TEAM_30G,FG3A_Open_diff_30G_Mavg,FG3A_Wide_Open_diff_30G_Mavg,FG3A_Tight_diff_30G_Mavg,FG3A_Very_Tight_diff_30G_Mavg
0,1610612737,Atlanta Hawks,0.0,2023-11-22,-0.923077,0.307692,-0.769231,-0.153846,1.0,-0.923077,0.307692,-0.769231,-0.153846
1,1610612737,Atlanta Hawks,0.0,2023-11-30,0.444444,1.055556,-0.277778,0.000000,2.0,-0.239316,0.681624,-0.523504,-0.076923
2,1610612737,Atlanta Hawks,0.0,2023-12-18,-0.769231,-1.846154,-0.076923,0.000000,3.0,-0.415954,-0.160969,-0.374644,-0.051282
3,1610612737,Atlanta Hawks,0.0,2024-01-15,-0.692308,2.512821,-0.153846,0.000000,4.0,-0.485043,0.507479,-0.319444,-0.038462
4,1610612737,Atlanta Hawks,0.0,2024-02-14,-0.933333,1.066667,1.266667,0.000000,5.0,-0.574701,0.619316,-0.002222,-0.030769
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15964,1610612766,Charlotte Hornets,5.0,2025-02-27,-0.616667,-1.500000,1.150000,-0.016667,30.0,-0.018519,0.033704,0.032870,0.009444
15965,1610612766,Charlotte Hornets,5.0,2025-03-03,-0.533333,4.100000,-1.133333,-0.150000,30.0,-0.102963,0.087037,-0.000741,0.004444
15966,1610612766,Charlotte Hornets,5.0,2025-03-05,-0.950000,0.066667,-0.950000,-0.100000,30.0,-0.117407,0.067593,-0.015741,0.001111
15967,1610612766,Charlotte Hornets,5.0,2025-03-07,-1.016667,-0.716667,-0.566667,-0.016667,30.0,-0.227963,-0.053519,-0.046296,0.001111


In [807]:

closestdefender_matchup.loc[(closestdefender_matchup['Cluster_3pt']==3)]['PLAYER_NAME'].unique()

array(['James Harden', 'Stephen Curry', 'Paul George', 'Kyrie Irving',
       'Damian Lillard', 'CJ McCollum', 'Zach LaVine', 'Devin Booker',
       'Fred VanVleet', 'Jayson Tatum', 'Donovan Mitchell',
       'Jalen Brunson', 'Shai Gilgeous-Alexander', 'Anfernee Simons',
       'Trae Young', 'Luka Dončić', 'Darius Garland', 'Tyler Herro',
       'Jordan Poole', 'Anthony Edwards', 'LaMelo Ball',
       'Tyrese Haliburton', 'Tyrese Maxey', 'Jalen Green', 'Cam Thomas',
       'Cade Cunningham', 'Keyonte George'], dtype=object)

In [808]:
gamelogs_def_closestdefender_avg_3pt.loc[(gamelogs_def_closestdefender_avg_3pt['OPPONENT_NAME']=='Atlanta Hawks')&(gamelogs_def_closestdefender_avg_3pt['Cluster_3pt']==3)]

,OPPONENT_ID,OPPONENT_NAME,Cluster_3pt,GAME_DATE,FG3A_Open_diff,FG3A_Wide_Open_diff,FG3A_Tight_diff,FG3A_Very_Tight_diff,GAMES_IN_WINDOW_TEAM_30G,FG3A_Open_diff_30G_Mavg,FG3A_Wide_Open_diff_30G_Mavg,FG3A_Tight_diff_30G_Mavg,FG3A_Very_Tight_diff_30G_Mavg
230,1610612737,Atlanta Hawks,3.0,2023-10-25,0.000000,0.000000,0.000000,0.000000,1.0,0.000000,0.000000,0.000000,0.000000
231,1610612737,Atlanta Hawks,3.0,2023-10-27,1.000000,1.500000,-0.500000,0.000000,2.0,0.500000,0.750000,-0.250000,0.000000
232,1610612737,Atlanta Hawks,3.0,2023-10-29,-1.000000,0.500000,-1.000000,0.000000,3.0,0.000000,0.666667,-0.500000,0.000000
233,1610612737,Atlanta Hawks,3.0,2023-10-30,0.000000,2.333333,-1.000000,0.000000,4.0,0.000000,1.083333,-0.625000,0.000000
234,1610612737,Atlanta Hawks,3.0,2023-11-01,0.000000,-1.500000,0.250000,0.000000,5.0,0.000000,0.566667,-0.450000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,1610612737,Atlanta Hawks,3.0,2025-02-24,-1.633333,2.066667,-0.700000,-0.166667,30.0,0.023713,0.566042,-0.133424,-0.038889
315,1610612737,Atlanta Hawks,3.0,2025-02-26,1.316667,-0.933333,-1.716667,-0.166667,30.0,0.145726,0.508889,-0.179188,-0.044444
316,1610612737,Atlanta Hawks,3.0,2025-02-28,0.933333,1.216667,-0.650000,0.000000,30.0,0.190726,0.426111,-0.100855,-0.044444
317,1610612737,Atlanta Hawks,3.0,2025-03-04,0.866667,1.100000,-1.633333,0.700000,30.0,0.245726,0.421111,-0.124744,-0.080556


In [809]:
gamelogs_def_closestdefender_avg['GAME_DATE'] = pd.to_datetime(gamelogs_def_closestdefender_avg['GAME_DATE'])
most_recent_mask = gamelogs_def_closestdefender_avg['GAME_DATE'] == gamelogs_def_closestdefender_avg.groupby('OPPONENT_ID')['GAME_DATE'].transform('max')
closest_def_60Day_new = pd.DataFrame(gamelogs_def_closestdefender_avg[most_recent_mask])

gamelogs_def_closestdefender_avg_pts['GAME_DATE'] = pd.to_datetime(gamelogs_def_closestdefender_avg_pts['GAME_DATE'])
most_recent_mask = gamelogs_def_closestdefender_avg_pts['GAME_DATE'] == gamelogs_def_closestdefender_avg_pts.groupby(['OPPONENT_ID','Cluster_Pts'])['GAME_DATE'].transform('max')
closest_def_60Day_new_pts = pd.DataFrame(gamelogs_def_closestdefender_avg_pts[most_recent_mask])

gamelogs_def_closestdefender_avg_3pt['GAME_DATE'] = pd.to_datetime(gamelogs_def_closestdefender_avg_3pt['GAME_DATE'])
most_recent_mask = gamelogs_def_closestdefender_avg_3pt['GAME_DATE'] == gamelogs_def_closestdefender_avg_3pt.groupby(['OPPONENT_ID','Cluster_3pt'])['GAME_DATE'].transform('max')
closest_def_60Day_new_3pt = pd.DataFrame(gamelogs_def_closestdefender_avg_3pt[most_recent_mask])

In [810]:
closest_def_60Day_new_pts

,OPPONENT_ID,OPPONENT_NAME,Cluster_Pts,GAME_DATE,FG3A_Open_diff,FGA_Open_diff,FG3A_Wide_Open_diff,FGA_Wide_Open_diff,FG3A_Tight_diff,FGA_Tight_diff,FG3A_Very_Tight_diff,FGA_Very_Tight_diff,GAMES_IN_WINDOW_TEAM_60G,FG3A_Open_diff_60G_Mavg,FGA_Open_diff_60G_Mavg,FG3A_Wide_Open_diff_60G_Mavg,FGA_Wide_Open_diff_60G_Mavg,FG3A_Tight_diff_60G_Mavg,FGA_Tight_diff_60G_Mavg,FG3A_Very_Tight_diff_60G_Mavg,FGA_Very_Tight_diff_60G_Mavg
621,1610612737,Atlanta Hawks,1.0,2025-03-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,60.0,0.557220,0.708683,0.145282,0.206672,-0.090787,0.044611,-0.016746,-0.060533
623,1610612737,Atlanta Hawks,5.0,2025-03-08,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,60.0,0.573053,0.730627,0.141671,0.199728,-0.072176,0.089889,-0.016468,-0.084699
625,1610612737,Atlanta Hawks,2.0,2025-03-10,-0.450000,-0.966667,-2.116667,-2.316667,0.783333,1.150000,0.000000,-0.516667,60.0,0.458609,0.650905,0.133894,0.189172,-0.090787,0.107111,-0.010913,-0.114144
627,1610612737,Atlanta Hawks,6.0,2025-03-10,0.650000,-0.083333,1.350000,1.150000,-0.666667,1.266667,0.933333,3.433333,60.0,0.484164,0.637016,0.097782,0.164728,-0.074676,0.271278,0.004365,-0.045255
628,1610612737,Atlanta Hawks,0.0,2025-03-12,-2.750000,-2.016667,4.000000,3.450000,-1.000000,-4.150000,-0.116667,2.366667,60.0,0.417776,0.561461,0.180005,0.245283,-0.071898,0.194333,0.007976,0.014467
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18816,1610612766,Charlotte Hornets,6.0,2025-03-10,-0.566667,-1.283333,-0.166667,-0.333333,-0.133333,0.433333,-0.016667,-0.816667,60.0,-0.078515,-0.342515,0.998807,0.960572,-0.135392,-0.660808,-0.016532,-0.107161
18817,1610612766,Charlotte Hornets,0.0,2025-03-12,-2.266667,0.183333,-0.450000,-1.050000,-0.666667,3.516667,-0.033333,-0.616667,60.0,-0.099904,-0.307793,0.992418,0.950017,-0.154003,-0.601919,-0.017088,-0.115217
18818,1610612766,Charlotte Hornets,1.0,2025-03-12,2.495977,0.798276,-2.517816,-3.448851,3.495977,-1.637931,-0.033908,-1.659770,60.0,-0.049693,-0.285321,0.966844,0.915869,-0.095181,-0.674774,-0.017653,-0.147880
18819,1610612766,Charlotte Hornets,2.0,2025-03-12,-0.566667,-2.316667,-1.366667,-1.783333,-0.083333,5.016667,0.000000,0.900000,60.0,0.004752,-0.270877,0.886288,0.794480,-0.113237,-0.490607,-0.017653,-0.132602


In [811]:
wndw = 30

closest_def_60Day_new# For Min-Max scaling to 0-2 range (centered at 1)
min_max_scaler = MinMaxScaler(feature_range=(0, 2))

# For Z-score normalization
standard_scaler = StandardScaler()
# Or using scipy
# z_scores = stats.zscore(df[columns])

# Columns to normalize
shot_diff_columns = [
    'FG3A_Open_diff_60G_Mavg', 'FGA_Open_diff_60G_Mavg',
    'FG3A_Wide_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg',
    'FG3A_Tight_diff_60G_Mavg', 'FGA_Tight_diff_60G_Mavg',
    'FG3A_Very_Tight_diff_60G_Mavg', 'FGA_Very_Tight_diff_60G_Mavg'
]

# Apply MinMaxScaler
normalized_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(closest_def_60Day_new[shot_diff_columns]),
    columns=[f"{col}_norm" for col in shot_diff_columns],
    index=closest_def_60Day_new.index
)

# Apply StandardScaler (Z-score)
normalized_zscore = pd.DataFrame(
    standard_scaler.fit_transform(closest_def_60Day_new[shot_diff_columns]),
    columns=[f"{col}_zscore" for col in shot_diff_columns],
    index=closest_def_60Day_new.index
)

# If you want z-scores centered at 1 instead of 0
normalized_zscore_centered = normalized_zscore + 1

# Combine with original dataframe if needed
df_normalized = pd.concat([closest_def_60Day_new, normalized_minmax, normalized_zscore_centered], axis=1)


closest_def_60Day_new_pts
# For Min-Max scaling to 0-2 range (centered at 1)
min_max_scaler = MinMaxScaler(feature_range=(0, 2))

# For Z-score normalization
standard_scaler = StandardScaler()
# Or using scipy
# z_scores = stats.zscore(df[columns])

# Columns to normalize
shot_diff_columns = [
    'FG3A_Open_diff_60G_Mavg', 'FGA_Open_diff_60G_Mavg',
    'FG3A_Wide_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg',
    'FG3A_Tight_diff_60G_Mavg', 'FGA_Tight_diff_60G_Mavg',
    'FG3A_Very_Tight_diff_60G_Mavg', 'FGA_Very_Tight_diff_60G_Mavg'
]

# Apply MinMaxScaler
normalized_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(closest_def_60Day_new_pts[shot_diff_columns]),
    columns=[f"{col}_norm" for col in shot_diff_columns],
    index=closest_def_60Day_new_pts.index
)

# Apply StandardScaler (Z-score)
normalized_zscore = pd.DataFrame(
    standard_scaler.fit_transform(closest_def_60Day_new_pts[shot_diff_columns]),
    columns=[f"{col}_zscore" for col in shot_diff_columns],
    index=closest_def_60Day_new_pts.index
)

# If you want z-scores centered at 1 instead of 0
normalized_zscore_centered = normalized_zscore + 1

# Combine with original dataframe if needed
df_normalized_pts = pd.concat([closest_def_60Day_new_pts, normalized_minmax, normalized_zscore_centered], axis=1)


#closest_def_60Day_new_3pt
# For Min-Max scaling to 0-2 range (centered at 1)
min_max_scaler = MinMaxScaler(feature_range=(0, 2))

# For Z-score normalization
standard_scaler = StandardScaler()
# Or using scipy
# z_scores = stats.zscore(df[columns])

# Columns to normalize
shot_diff_columns = [
    f'FG3A_Open_diff_{wndw}G_Mavg', f'FG3A_Wide_Open_diff_{wndw}G_Mavg',f'FG3A_Tight_diff_{wndw}G_Mavg', f'FG3A_Very_Tight_diff_{wndw}G_Mavg'
]

# Apply MinMaxScaler
normalized_minmax = pd.DataFrame(
    min_max_scaler.fit_transform(closest_def_60Day_new_3pt[shot_diff_columns]),
    columns=[f"{col}_norm" for col in shot_diff_columns],
    index=closest_def_60Day_new_3pt.index
)

# Apply StandardScaler (Z-score)
normalized_zscore = pd.DataFrame(
    standard_scaler.fit_transform(closest_def_60Day_new_3pt[shot_diff_columns]),
    columns=[f"{col}_zscore" for col in shot_diff_columns],
    index=closest_def_60Day_new_3pt.index
)

# If you want z-scores centered at 1 instead of 0
normalized_zscore_centered = normalized_zscore + 1

# Combine with original dataframe if needed
df_normalized_3pt = pd.concat([closest_def_60Day_new_3pt, normalized_minmax, normalized_zscore_centered], axis=1)

In [812]:
#df_normalized.columns

In [813]:
#closest_def_60Day_new.head(5)

In [814]:
closest_def_ratio = pd.DataFrame(df_normalized[['OPPONENT_ID', 'OPPONENT_NAME', 'FG3A_Open_diff_60G_Mavg_norm', 'FGA_Open_diff_60G_Mavg_norm',
       'FG3A_Wide_Open_diff_60G_Mavg_norm', 'FGA_Wide_Open_diff_60G_Mavg_norm',
       'FG3A_Tight_diff_60G_Mavg_norm', 'FGA_Tight_diff_60G_Mavg_norm',
       'FG3A_Very_Tight_diff_60G_Mavg_norm',
       'FGA_Very_Tight_diff_60G_Mavg_norm', 'FG3A_Open_diff_60G_Mavg_zscore',
       'FGA_Open_diff_60G_Mavg_zscore', 'FG3A_Wide_Open_diff_60G_Mavg_zscore',
       'FGA_Wide_Open_diff_60G_Mavg_zscore', 'FG3A_Tight_diff_60G_Mavg_zscore',
       'FGA_Tight_diff_60G_Mavg_zscore',
       'FG3A_Very_Tight_diff_60G_Mavg_zscore',
       'FGA_Very_Tight_diff_60G_Mavg_zscore']])

closest_def_pts_ratio = pd.DataFrame(df_normalized_pts[['OPPONENT_ID', 'OPPONENT_NAME','Cluster_Pts', 'FG3A_Open_diff_60G_Mavg_norm', 'FGA_Open_diff_60G_Mavg_norm',
       'FG3A_Wide_Open_diff_60G_Mavg_norm', 'FGA_Wide_Open_diff_60G_Mavg_norm',
       'FG3A_Tight_diff_60G_Mavg_norm', 'FGA_Tight_diff_60G_Mavg_norm',
       'FG3A_Very_Tight_diff_60G_Mavg_norm',
       'FGA_Very_Tight_diff_60G_Mavg_norm', 'FG3A_Open_diff_60G_Mavg_zscore',
       'FGA_Open_diff_60G_Mavg_zscore', 'FG3A_Wide_Open_diff_60G_Mavg_zscore',
       'FGA_Wide_Open_diff_60G_Mavg_zscore', 'FG3A_Tight_diff_60G_Mavg_zscore',
       'FGA_Tight_diff_60G_Mavg_zscore',
       'FG3A_Very_Tight_diff_60G_Mavg_zscore',
       'FGA_Very_Tight_diff_60G_Mavg_zscore']])

closest_def_3pt_ratio = pd.DataFrame(df_normalized_3pt[['OPPONENT_ID', 'OPPONENT_NAME','Cluster_3pt', f'FG3A_Open_diff_{wndw}G_Mavg_norm', 
       f'FG3A_Wide_Open_diff_{wndw}G_Mavg_norm', f'FG3A_Tight_diff_{wndw}G_Mavg_norm', 
       f'FG3A_Very_Tight_diff_{wndw}G_Mavg_norm', f'FG3A_Open_diff_{wndw}G_Mavg_zscore',f'FG3A_Wide_Open_diff_{wndw}G_Mavg_zscore',
        f'FG3A_Tight_diff_{wndw}G_Mavg_zscore',f'FG3A_Very_Tight_diff_{wndw}G_Mavg_zscore']])

In [815]:
closest_def_ratio.head(5)

,OPPONENT_ID,OPPONENT_NAME,FG3A_Open_diff_60G_Mavg_norm,FGA_Open_diff_60G_Mavg_norm,FG3A_Wide_Open_diff_60G_Mavg_norm,FGA_Wide_Open_diff_60G_Mavg_norm,FG3A_Tight_diff_60G_Mavg_norm,FGA_Tight_diff_60G_Mavg_norm,FG3A_Very_Tight_diff_60G_Mavg_norm,FGA_Very_Tight_diff_60G_Mavg_norm,FG3A_Open_diff_60G_Mavg_zscore,FGA_Open_diff_60G_Mavg_zscore,FG3A_Wide_Open_diff_60G_Mavg_zscore,FGA_Wide_Open_diff_60G_Mavg_zscore,FG3A_Tight_diff_60G_Mavg_zscore,FGA_Tight_diff_60G_Mavg_zscore,FG3A_Very_Tight_diff_60G_Mavg_zscore,FGA_Very_Tight_diff_60G_Mavg_zscore
146,1610612737,Atlanta Hawks,1.150683,1.152857,1.000568,0.960295,1.033279,1.263781,1.306673,1.235397,1.809838,1.682063,0.698153,0.737757,0.984334,1.290944,1.796232,1.214292
294,1610612738,Boston Celtics,0.186446,0.558511,1.243315,1.276337,1.113613,1.183931,1.715409,0.815001,-0.521745,0.407614,1.310207,1.536748,1.152512,1.128331,2.587013,0.316092
441,1610612739,Cleveland Cavaliers,0.979457,1.780740,0.902646,0.844390,1.510600,1.860420,1.387468,1.209000,1.395805,3.028425,0.451255,0.444736,1.983598,2.505985,1.952545,1.157892
589,1610612740,New Orleans Pelicans,1.036704,0.560825,1.798041,1.581382,0.552971,0.656759,1.011253,1.250965,1.534230,0.412577,2.708875,2.307936,-0.021185,0.054757,1.224683,1.247553
736,1610612741,Chicago Bulls,1.486598,1.425393,0.930312,0.857997,0.000000,1.282127,0.733021,1.533332,2.622101,2.266459,0.521013,0.479136,-1.178822,1.328305,0.686388,1.850846


In [816]:
def calculate_defensive_metrics(df, open3_weight=3.3, wo3_weight=4.5, tight3_weight=.8, vt3_weight=.01, open_weight=5, wo_weight=3, tight_weight=5, vt_weight=3):
    """
    Calculate defensive metrics based on shot type ratios.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing defensive statistics
    open_weight (int): Weight for open shots (default: 5)
    wo_weight (int): Weight for wide open shots (default: 3)
    tight_weight (int): Weight for tight shots (default: 5)
    vt_weight (int): Weight for very tight shots (default: 3)
    
    Returns:
    pandas.DataFrame: DataFrame with additional defensive metric columns
    """
    # Create a copy to avoid modifying the original DataFrame
    result_df = df.copy()
    
    # Calculate weights denominators
    open3_total_weight = open3_weight + wo3_weight
    tight3_total_weight = tight3_weight + vt3_weight

    open_total_weight = open_weight + wo_weight
    tight_total_weight = tight_weight + vt_weight
    
    # 3-Point Shot Metrics
    open3_norm_score = (
        (result_df['FG3A_Open_diff_60G_Mavg_norm'] * open3_weight) + 
        (result_df['FG3A_Wide_Open_diff_60G_Mavg_norm'] * wo3_weight)
    ) / open_total_weight
    
    tight3_norm_score = (
        (result_df['FG3A_Tight_diff_60G_Mavg_norm'] * tight3_weight) + 
        (result_df['FG3A_Very_Tight_diff_60G_Mavg_norm'] * vt3_weight)
    ) / tight_total_weight
    
    open3_zscore = (
        (result_df['FG3A_Open_diff_60G_Mavg_zscore'] * open3_weight) + 
        (result_df['FG3A_Wide_Open_diff_60G_Mavg_zscore'] * wo3_weight)
    ) / open_total_weight
    
    tight3_zscore = (
        (result_df['FG3A_Tight_diff_60G_Mavg_zscore'] * tight3_weight) + 
        (result_df['FG3A_Very_Tight_diff_60G_Mavg_zscore'] * vt3_weight)
    ) / tight_total_weight
    
    # All Field Goal Attempts Metrics
    open_norm_score = (
        (result_df['FGA_Open_diff_60G_Mavg_norm'] * open_weight) + 
        (result_df['FGA_Wide_Open_diff_60G_Mavg_norm'] * wo_weight)
    ) / open_total_weight
    
    tight_norm_score = -(
        (result_df['FGA_Tight_diff_60G_Mavg_norm'] * tight_weight) + 
        (result_df['FGA_Very_Tight_diff_60G_Mavg_norm'] * vt_weight)
    ) / tight_total_weight
    
    open_zscore = (
        (result_df['FGA_Open_diff_60G_Mavg_zscore'] * open_weight) + 
        (result_df['FGA_Wide_Open_diff_60G_Mavg_zscore'] * wo_weight)
    ) / open_total_weight
    
    tight_zscore = -(
        (result_df['FGA_Tight_diff_60G_Mavg_zscore'] * tight_weight) + 
        (result_df['FGA_Very_Tight_diff_60G_Mavg_zscore'] * vt_weight)
    ) / tight_total_weight
    
    # Calculate final scores
    result_df['defense_norm_score'] = (((open_norm_score + tight_norm_score) / 2) * 10).round(2)
    result_df['defense_zscore'] = (((open_zscore + tight_zscore) / 2) * 10).round(2)
    result_df['defense3_norm_score'] = (((open3_norm_score + tight3_norm_score) / 2) * 10).round(2)
    result_df['defense3_zscore'] = (((open3_zscore + tight3_zscore) / 2) * 10).round(2)
    
    return result_df


def calculate_defensive_metrics_3pt(df, open_weight=3.3, wo_weight=4.5, tight_weight=0.8, vt_weight=.01):
    """
    Calculate defensive metrics based on 3-point shot type ratios.
    
    Parameters:
    df (pandas.DataFrame): DataFrame containing 3-point defensive statistics
    open_weight (int): Weight for open shots (default: 5)
    wo_weight (int): Weight for wide open shots (default: 3)
    tight_weight (int): Weight for tight shots (default: 5)
    vt_weight (int): Weight for very tight shots (default: 3)
    
    Returns:
    pandas.DataFrame: DataFrame with additional defensive metric columns
    """
    # Create a copy to avoid modifying the original DataFrame
    result_df = df.copy()
    
    # Calculate weights denominators
    open_total_weight = open_weight + wo_weight
    tight_total_weight = tight_weight + vt_weight
    
    # 3-Point Shot Metrics
    open3_norm_score = (
        (result_df['FG3A_Open_diff_60G_Mavg_norm'] * open_weight) + 
        (result_df['FG3A_Wide_Open_diff_60G_Mavg_norm'] * wo_weight)
    ) / open_total_weight
    
    tight3_norm_score = (
        (result_df['FG3A_Tight_diff_60G_Mavg_norm'] * tight_weight) + 
        (result_df['FG3A_Very_Tight_diff_60G_Mavg_norm'] * vt_weight)
    ) / tight_total_weight
    
    open3_zscore = (
        (result_df['FG3A_Open_diff_60G_Mavg_zscore'] * open_weight) + 
        (result_df['FG3A_Wide_Open_diff_60G_Mavg_zscore'] * wo_weight)
    ) / open_total_weight
    
    tight3_zscore = (
        (result_df['FG3A_Tight_diff_60G_Mavg_zscore'] * tight_weight) + 
        (result_df['FG3A_Very_Tight_diff_60G_Mavg_zscore'] * vt_weight)
    ) / tight_total_weight

    result_df['open_defense_score'] = (
        (result_df['FG3A_Open_diff_60G_Mavg_zscore'] * open_weight) + 
        (result_df['FG3A_Wide_Open_diff_60G_Mavg_zscore'] * wo_weight)
    ) / open_total_weight
    
    result_df['tight_defense_score'] = (
        (result_df['FG3A_Tight_diff_60G_Mavg_zscore'] * tight_weight) + 
        (result_df['FG3A_Very_Tight_diff_60G_Mavg_zscore'] * vt_weight)
    ) / tight_total_weight

    #result_df['defense3_score'] = (result_df['open_defense_score'] + result_df['tight_defense_score']) / 2
    
    # Add interpretive columns
    result_df['open_defense_rating'] = pd.qcut(result_df['open_defense_score'], q=5, labels=['Elite', 'Good', 'Average', 'Below Average', 'Poor'])
    result_df['tight_defense_rating'] = pd.qcut(result_df['tight_defense_score'], q=5, labels=['Elite', 'Good', 'Average', 'Below Average', 'Poor'])
    
    # Calculate final scores
    result_df['defense3_norm_score'] = (((open3_norm_score + tight3_norm_score) / 2) * 10).round(2)
    result_df['defense3_zscore'] = (((open3_zscore + tight3_zscore) / 2) * 10).round(2)
    
    return result_df

def calculate_defensive_metrics_3pt(df, open_weight=3.3, wo_weight=4.5, tight_weight=0.8, vt_weight=.01):
    result_df = df.copy()
    
    # Calculate league averages for each metric
    league_avgs = {
        'open': df[f'FG3A_Open_diff_{wndw}G_Mavg'].mean(),
        'wide_open': df[f'FG3A_Wide_Open_diff_{wndw}G_Mavg'].mean(),
        'tight': df[f'FG3A_Tight_diff_{wndw}G_Mavg'].mean(),
        'very_tight': df[f'FG3A_Very_Tight_diff_{wndw}G_Mavg'].mean()
    }
    
    # Calculate percentage differences from league average
    result_df['open_vs_avg'] = (df[f'FG3A_Open_diff_{wndw}G_Mavg'] - league_avgs['open']) 
    result_df['wide_open_vs_avg'] = (df[f'FG3A_Wide_Open_diff_{wndw}G_Mavg'] - league_avgs['wide_open']) 
    result_df['tight_vs_avg'] = (df[f'FG3A_Tight_diff_{wndw}G_Mavg'] - league_avgs['tight'])
    result_df['very_tight_vs_avg'] = (df[f'FG3A_Very_Tight_diff_{wndw}G_Mavg'] - league_avgs['very_tight'])
    
    # Calculate weighted scores for open and tight shots
    open_total_weight = open_weight + wo_weight
    tight_total_weight = tight_weight + vt_weight

    result_df['league_wide_open'] = league_avgs['wide_open']
        
    result_df['open_defense_score'] = (
        (result_df['open_vs_avg'] * open_weight + 
         result_df['wide_open_vs_avg'] * wo_weight) / open_total_weight
    )
    
    result_df['tight_defense_score'] = (
        (result_df['tight_vs_avg'] * tight_weight + 
         result_df['very_tight_vs_avg'] * vt_weight) / tight_total_weight
    )
    
    # Calculate final defense score
    # Negative scores mean better defense (reducing shots vs average)
    result_df['defense3_score'] = (result_df['open_defense_score'] + result_df['tight_defense_score']) / 2
    
    # Add interpretive columns
    result_df['open_defense_rating'] = pd.qcut(result_df['open_defense_score'], q=5, labels=['Elite', 'Good', 'Average', 'Below Average', 'Poor'])
    result_df['tight_defense_rating'] = pd.qcut(result_df['tight_defense_score'], q=5, labels=['Elite', 'Good', 'Average', 'Below Average', 'Poor'])
    
    return result_df

# Calculate for each cluster
def calculate_cluster_defense_metrics(df):
    cluster_metrics = []
    
    for cluster in df['Cluster_3pt'].unique():
        cluster_df = df[df['Cluster_3pt'] == cluster].copy()
        cluster_results = calculate_defensive_metrics_3pt(cluster_df)
        cluster_metrics.append(cluster_results)
    
    return pd.concat(cluster_metrics)

# Apply calculations
cluster_3pt_closest_def_scores = calculate_cluster_defense_metrics(closest_def_60Day_new_3pt)

In [817]:
cluster_3pt_closest_def_scores

,OPPONENT_ID,OPPONENT_NAME,Cluster_3pt,GAME_DATE,FG3A_Open_diff,FG3A_Wide_Open_diff,FG3A_Tight_diff,FG3A_Very_Tight_diff,GAMES_IN_WINDOW_TEAM_30G,FG3A_Open_diff_30G_Mavg,FG3A_Wide_Open_diff_30G_Mavg,FG3A_Tight_diff_30G_Mavg,FG3A_Very_Tight_diff_30G_Mavg,open_vs_avg,wide_open_vs_avg,tight_vs_avg,very_tight_vs_avg,league_wide_open,open_defense_score,tight_defense_score,defense3_score,open_defense_rating,tight_defense_rating
10,1610612737,Atlanta Hawks,0.0,2025-03-10,0.650000,1.350000,-0.666667,0.933333,11.0,-0.007144,0.369104,-0.052042,0.045592,-0.152899,0.477390,-0.126608,0.031389,-0.108287,0.210729,-0.124658,0.043036,Below Average,Good
546,1610612738,Boston Celtics,0.0,2025-03-06,3.750000,1.350000,0.333333,-0.050000,14.0,-0.096004,-0.389885,0.582424,-0.032686,-0.241759,-0.281598,0.507858,-0.046889,-0.108287,-0.264743,0.501010,0.118133,Good,Poor
1082,1610612739,Cleveland Cavaliers,0.0,2025-03-07,-1.222222,-0.555556,-0.111111,0.000000,11.0,-0.010315,-0.749500,-0.062533,-0.025553,-0.156070,-0.641214,-0.137098,-0.039757,-0.108287,-0.435961,-0.135897,-0.285929,Elite,Good
1619,1610612740,New Orleans Pelicans,0.0,2024-04-05,3.588235,2.352941,3.235294,0.000000,8.0,0.788775,0.364849,0.336270,-0.020833,0.643020,0.473136,0.261705,-0.035037,-0.108287,0.545010,0.258041,0.401526,Poor,Below Average
2148,1610612741,Chicago Bulls,0.0,2025-02-24,0.830508,1.254237,0.288136,-0.033898,11.0,0.438039,-0.150477,-0.243022,0.069277,0.292284,-0.042190,-0.317588,0.055074,-0.108287,0.099318,-0.312987,-0.106835,Average,Elite
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13840,1610612762,Utah Jazz,5.0,2025-03-10,0.900000,2.650000,-1.433333,-0.016667,30.0,0.253889,1.866667,-0.275556,-0.006111,0.136117,1.865758,-0.321990,-0.014407,0.000909,1.133987,-0.318193,0.407897,Poor,Elite
14383,1610612763,Memphis Grizzlies,5.0,2025-03-12,-0.400000,0.233333,0.950000,-0.016667,30.0,0.079444,0.336111,-0.032778,0.010000,-0.038328,0.335202,-0.079212,0.001704,0.000909,0.177170,-0.078213,0.049479,Below Average,Good
14916,1610612764,Washington Wizards,5.0,2025-03-13,-3.816667,-2.450000,0.383333,-0.300000,30.0,0.041111,0.787778,-0.109444,-0.047778,-0.076661,0.786869,-0.155879,-0.056074,0.000909,0.421529,-0.154647,0.133441,Poor,Good
15438,1610612765,Detroit Pistons,5.0,2025-03-09,1.850000,1.316667,1.983333,-0.016667,30.0,0.777222,0.327222,0.127222,-0.036111,0.659450,0.326314,0.080788,-0.044407,0.000909,0.467256,0.079242,0.273249,Poor,Below Average


In [818]:
# Or with custom weights
total_closest_def_scores = calculate_defensive_metrics(
    closest_def_ratio,
)
cluster_pts_closest_def_scores = calculate_defensive_metrics(
    closest_def_pts_ratio,
)
#cluster_3pt_closest_def_scores = calculate_defensive_metrics_3pt(
#    closest_def_3pt_ratio,
#)


In [819]:
#total_closest_def_scores.head()

In [820]:
closest_def_pts_score = closest_def_60Day_new_pts[['OPPONENT_ID','OPPONENT_NAME','Cluster_Pts', 'FGA_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg','FGA_Tight_diff_60G_Mavg','FGA_Very_Tight_diff_60G_Mavg',
                                'FG3A_Open_diff_60G_Mavg','FG3A_Wide_Open_diff_60G_Mavg','FG3A_Tight_diff_60G_Mavg','FG3A_Very_Tight_diff_60G_Mavg',]].merge(cluster_pts_closest_def_scores[['OPPONENT_ID','OPPONENT_NAME','Cluster_Pts','defense_norm_score','defense_zscore',
                                'defense3_norm_score','defense3_zscore']],how='left')

closest_def_total_score = closest_def_60Day_new[['OPPONENT_ID','OPPONENT_NAME', 'FGA_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg','FGA_Tight_diff_60G_Mavg','FGA_Very_Tight_diff_60G_Mavg',
                                'FG3A_Open_diff_60G_Mavg','FG3A_Wide_Open_diff_60G_Mavg','FG3A_Tight_diff_60G_Mavg','FG3A_Very_Tight_diff_60G_Mavg',]].merge(total_closest_def_scores[['OPPONENT_ID','OPPONENT_NAME','defense_norm_score','defense_zscore',
                                'defense3_norm_score','defense3_zscore']],how='left')

closest_def_3pt_score = closest_def_60Day_new_3pt[['OPPONENT_ID','OPPONENT_NAME','Cluster_3pt', f'FG3A_Open_diff_{wndw}G_Mavg',f'FG3A_Wide_Open_diff_{wndw}G_Mavg',f'FG3A_Tight_diff_{wndw}G_Mavg',
                                                   f'FG3A_Very_Tight_diff_{wndw}G_Mavg',]].merge(cluster_3pt_closest_def_scores[['OPPONENT_ID','OPPONENT_NAME','Cluster_3pt','open_defense_score','tight_defense_score','open_defense_rating','tight_defense_rating','defense3_score']],how='left')

In [821]:
closest_def_pts_df = pd.DataFrame(closest_def_pts_score[['OPPONENT_ID','OPPONENT_NAME','Cluster_Pts', 'FGA_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg','FGA_Tight_diff_60G_Mavg','FGA_Very_Tight_diff_60G_Mavg','defense_norm_score','defense_zscore',
                                'FG3A_Open_diff_60G_Mavg','FG3A_Wide_Open_diff_60G_Mavg','FG3A_Tight_diff_60G_Mavg','FG3A_Very_Tight_diff_60G_Mavg','defense3_norm_score','defense3_zscore']])
closest_def_total_df = pd.DataFrame(closest_def_total_score[['OPPONENT_ID','OPPONENT_NAME', 'FGA_Open_diff_60G_Mavg', 'FGA_Wide_Open_diff_60G_Mavg','FGA_Tight_diff_60G_Mavg','FGA_Very_Tight_diff_60G_Mavg','defense_norm_score','defense_zscore',
                                'FG3A_Open_diff_60G_Mavg','FG3A_Wide_Open_diff_60G_Mavg','FG3A_Tight_diff_60G_Mavg','FG3A_Very_Tight_diff_60G_Mavg','defense3_norm_score','defense3_zscore']])

closest_def_3pt_df = pd.DataFrame(closest_def_3pt_score[['OPPONENT_ID','OPPONENT_NAME','Cluster_3pt', f'FG3A_Open_diff_{wndw}G_Mavg',f'FG3A_Wide_Open_diff_{wndw}G_Mavg',f'FG3A_Tight_diff_{wndw}G_Mavg',
                                                   f'FG3A_Very_Tight_diff_{wndw}G_Mavg','open_defense_score','tight_defense_score','open_defense_rating','tight_defense_rating','defense3_score']])

In [822]:
closest_def_pts_df['as_of'] = pd.to_datetime(today)
closest_def_pts_df['id'] = closest_def_pts_df['as_of'].astype(str)+"_"+closest_def_pts_df['OPPONENT_ID'].astype(str)+closest_def_pts_df['Cluster_Pts'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "closest_def_pts"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    closest_def_pts_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = closest_def_pts_df[~closest_def_pts_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'closest_def_pts' already exists. Checking for new records...
Inserted 210 new records into 'closest_def_pts'.


In [823]:
closest_def_total_df['as_of'] = pd.to_datetime(today)
closest_def_total_df['id'] = closest_def_total_df['as_of'].astype(str)+"_"+closest_def_total_df['OPPONENT_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "closest_def_total"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    closest_def_total_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = closest_def_total_df[~closest_def_total_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'closest_def_total' already exists. Checking for new records...
Inserted 30 new records into 'closest_def_total'.


In [824]:
closest_def_3pt_df['as_of'] = pd.to_datetime(today)
closest_def_3pt_df['id'] = closest_def_3pt_df['as_of'].astype(str)+"_"+closest_def_3pt_df['OPPONENT_ID'].astype(str)+closest_def_3pt_df['Cluster_3pt'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "closest_def_3pt"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    closest_def_3pt_df.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = closest_def_3pt_df[~closest_def_3pt_df['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'closest_def_3pt' already exists. Checking for new records...
Inserted 180 new records into 'closest_def_3pt'.


In [825]:
df_playtypes.columns

Index(['OPPONENT_ID', 'OPPONENT_NAME', 'OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition',
       'OVERALL_DEF_SCORE_PERCENTILE_Cut',
       'OVERALL_DEF_SCORE_PERCENTILE_Handoff',
       'OVERALL_DEF_SCORE_PERCENTILE_Isolation',
       'OVERALL_DEF_SCORE_PERCENTILE_Misc',
       'OVERALL_DEF_SCORE_PERCENTILE_OffRebound',
       'OVERALL_DEF_SCORE_PERCENTILE_OffScreen',
       'OVERALL_DEF_SCORE_PERCENTILE_PRBallHandler',
       'OVERALL_DEF_SCORE_PERCENTILE_PRRollMan',
       'OVERALL_DEF_SCORE_PERCENTILE_Postup',
       'OVERALL_DEF_SCORE_PERCENTILE_Spotup',
       'OVERALL_DEF_SCORE_PERCENTILE_Transition', 'DEF_EFFICIENCY_SCORE_Cut',
       'DEF_EFFICIENCY_SCORE_Handoff', 'DEF_

In [826]:
df_3s_stats = pd.DataFrame(df_playtypes[['OPPONENT_ID','OPPONENT_NAME','MIN','FG3A','OVERALL_DEF_SCORE_Handoff','DEF_EFFICIENCY_SCORE_Handoff','DISRUPTION_SCORE_Handoff','OVERALL_DEF_SCORE_OffScreen','DEF_EFFICIENCY_SCORE_OffScreen','DISRUPTION_SCORE_OffScreen',
                                       'OVERALL_DEF_SCORE_Isolation','DEF_EFFICIENCY_SCORE_Isolation','DISRUPTION_SCORE_Isolation','OVERALL_DEF_SCORE_PRBallHandler','DEF_EFFICIENCY_SCORE_PRBallHandler','DISRUPTION_SCORE_PRBallHandler',
                                       'OVERALL_DEF_SCORE_PRRollMan','DEF_EFFICIENCY_SCORE_PRRollMan','DISRUPTION_SCORE_PRRollMan','OVERALL_DEF_SCORE_Spotup','DEF_EFFICIENCY_SCORE_Spotup','DISRUPTION_SCORE_Spotup',
                                         'OVERALL_DEF_SCORE_Handoff','DEF_EFFICIENCY_SCORE_Handoff','DISRUPTION_SCORE_Handoff',
            'PU_FG3_PCT_60Day','PU_FG3A_rate_60Day','CS_FG3_PCT_60Day','CS_FG3A_rate_60Day','FGA_3_AB_60G_Sum', 'FGA_3_LC_60G_Sum',
       'FGA_3_RC_60G_Sum', 'FGM_3_AB_60G_Sum',
       'FGM_3_LC_60G_Sum', 'FGM_3_RC_60G_Sum']])

In [827]:
df_3s_stats['FG3A_Corner_Rate'] = df_3s_stats['FG3A']/(df_3s_stats['FGA_3_RC_60G_Sum']+df_3s_stats['FGA_3_LC_60G_Sum'])
df_3s_stats['Corner_FG_PCT'] = (df_3s_stats['FGM_3_RC_60G_Sum']+df_3s_stats['FGM_3_LC_60G_Sum'])/(df_3s_stats['FGA_3_RC_60G_Sum']+df_3s_stats['FGA_3_LC_60G_Sum'])
df_3s_stats['Corner_FGA_rate'] = (df_3s_stats['FGA_3_RC_60G_Sum']+df_3s_stats['FGA_3_LC_60G_Sum'])/df_3s_stats['MIN']

In [828]:
df_3s_stats.fillna(0, inplace=True)

In [829]:
league_avg_fg_pct = df_3s_stats['CS_FG3_PCT_60Day'].mean()
league_avg_fga_per_min = df_3s_stats['CS_FG3A_rate_60Day'].mean()
league_avg_screen_fg_pct = df_3s_stats['OVERALL_DEF_SCORE_OffScreen'].mean()
league_avg_pts_per_fgm_pnrman = df_3s_stats['OVERALL_DEF_SCORE_PRRollMan'].mean()
league_avg_pts_per_fgm_ho = df_3s_stats['OVERALL_DEF_SCORE_Handoff'].mean()
league_avg_pts_per_fgm_pnrball = df_3s_stats['OVERALL_DEF_SCORE_PRBallHandler'].mean()
#league_avg_poss_pct_pnr = df_3s_stats['POSS_PCT_PRRollMan'].mean()
league_avg_fg_pct_PU = df_3s_stats['PU_FG3_PCT_60Day'].mean()
league_avg_fga_per_min_PU = df_3s_stats['PU_FG3A_rate_60Day'].mean()
league_avg_iso_fg_pct = df_3s_stats['OVERALL_DEF_SCORE_Isolation'].mean()
league_avg_spotup_fg_pct = df_3s_stats['OVERALL_DEF_SCORE_Spotup'].mean()
league_avg_corner_fg_pct = df_3s_stats['Corner_FG_PCT'].mean()
league_avg_corner_fga_per_min = df_3s_stats['Corner_FGA_rate'].mean()

In [830]:


offscreen_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_OffScreen'] / league_avg_screen_fg_pct) # Scale by frequency to reward volume
    )
iso_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_Isolation'] / league_avg_iso_fg_pct)   # Scale by frequency to reward volume
    )
spotup_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_Spotup'] / league_avg_spotup_fg_pct)   # Scale by frequency to reward volume
    )

pnr_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_PRRollMan']/league_avg_pts_per_fgm_pnrman)
    # Scale by frequency to reward volume
    )
pnr_ball_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_PRBallHandler']/league_avg_pts_per_fgm_pnrball)
    # Scale by frequency to reward volume
    )
ho_score = (
        (df_3s_stats['OVERALL_DEF_SCORE_Handoff']/league_avg_pts_per_fgm_ho)
    # Scale by frequency to reward volume
    )

shooting_component = (
        (df_3s_stats['CS_FG3_PCT_60Day'] / league_avg_fg_pct) * 
        (df_3s_stats['CS_FG3A_rate_60Day'] / league_avg_fga_per_min))
shooting_component_pu = (
        (df_3s_stats['PU_FG3_PCT_60Day'] / league_avg_fg_pct_PU) * 
        (df_3s_stats['PU_FG3A_rate_60Day'] / league_avg_fga_per_min_PU))
shooting_component_corner = (
        (df_3s_stats['Corner_FG_PCT'] / league_avg_corner_fg_pct) * 
        (df_3s_stats['Corner_FGA_rate'] / league_avg_corner_fga_per_min))

In [831]:
df_3s_stats['offscreen_cse_score'] = (
        shooting_component * 0.5 +  # 60% weight on shooting
        offscreen_score * 0.3 +
        spotup_score * 0.2 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN']) * 100
average_score = df_3s_stats['offscreen_cse_score'].mean()
df_3s_stats['offscreen_cse_score'] = (df_3s_stats['offscreen_cse_score'] / average_score) * 100

df_3s_stats['pnp_cse_score'] = (
        shooting_component * 0.55 +  # 60% weight on shooting
        pnr_score * 0.30 +  # 60% weight on shooting
        spotup_score * 0.15 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN']) * 100
average_score = df_3s_stats['pnp_cse_score'].mean()
df_3s_stats['pnp_cse_score'] = (df_3s_stats['pnp_cse_score'] / average_score) * 100


df_3s_stats['iso_pue_score'] = (
        shooting_component_pu * 0.6 +  # 60% weight on shooting
        iso_score * 0.3 +
        pnr_ball_score * 0.1 # 40% weight on movement
    ) * np.log1p(df_3s_stats['MIN']) * 100
average_score = df_3s_stats['iso_pue_score'].mean()
df_3s_stats['iso_pue_score'] = (df_3s_stats['iso_pue_score'] / average_score) * 100

df_3s_stats['shooting_component']  = shooting_component
df_3s_stats['shooting_component_pu']  = shooting_component_pu
df_3s_stats['shooting_component_corner']  = shooting_component_corner

In [832]:
df_3s_stats['cse_score'] = (
        (df_3s_stats['CS_FG3_PCT_60Day'] / league_avg_fg_pct) * 
        (df_3s_stats['CS_FG3A_rate_60Day'] / league_avg_fga_per_min) * 
        np.log1p(df_3s_stats['MIN'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['cse_score'].mean()
df_3s_stats['cse_score'] = (df_3s_stats['cse_score'] / average_score) * 100

df_3s_stats['CnS_Rating'] = (df_3s_stats['CS_FG3A_rate_60Day'] * 
                              df_3s_stats['CS_FG3_PCT_60Day'] * 
                              np.sqrt(df_3s_stats['MIN'])).round(3)
df_3s_stats['pue_score'] = (
        (df_3s_stats['PU_FG3_PCT_60Day'] / league_avg_fg_pct_PU) * 
        (df_3s_stats['PU_FG3A_rate_60Day'] / league_avg_fga_per_min_PU) * 
        np.log1p(df_3s_stats['MIN'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['pue_score'].mean()
df_3s_stats['pue_score'] = (df_3s_stats['pue_score'] / average_score) * 100

df_3s_stats['PU_Rating'] = (df_3s_stats['PU_FG3A_rate_60Day'] * 
                              df_3s_stats['PU_FG3_PCT_60Day'] * 
                              np.sqrt(df_3s_stats['MIN'])).round(3)

df_3s_stats['corner_score'] = (
        (df_3s_stats['Corner_FG_PCT'] / league_avg_corner_fg_pct) * 
        (df_3s_stats['Corner_FGA_rate'] / league_avg_corner_fga_per_min) * 
        np.log1p(df_3s_stats['MIN'])  # Log transformation reduces extreme values
    ) * 100
average_score = df_3s_stats['corner_score'].mean()
df_3s_stats['corner_score'] = (df_3s_stats['corner_score'] / average_score) * 100


In [833]:
def_3s_ = pd.DataFrame(df_3s_stats[['OPPONENT_ID','OPPONENT_NAME','cse_score','offscreen_cse_score','pnp_cse_score','pue_score','iso_pue_score','corner_score']])

In [834]:
#def_3s_.sort_values(by='OVERALL_DEF_SCORE_OffScreen')

In [835]:
def_3s_['as_of'] = pd.to_datetime(today)
def_3s_['id'] = def_3s_['as_of'].astype(str)+"_"+def_3s_['OPPONENT_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "def_3pt_scores"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    def_3s_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = def_3s_[~def_3s_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'def_3pt_scores' already exists. Checking for new records...
Inserted 30 new records into 'def_3pt_scores'.


In [836]:
df_ast_cat = pd.DataFrame(df_playtypes[['OPPONENT_ID','OPPONENT_NAME','MIN','FGA','FG3A','OVERALL_DEF_SCORE_Cut',
       'OVERALL_DEF_SCORE_Handoff', 'OVERALL_DEF_SCORE_Isolation',
       'OVERALL_DEF_SCORE_Misc', 'OVERALL_DEF_SCORE_OffRebound',
       'OVERALL_DEF_SCORE_OffScreen', 'OVERALL_DEF_SCORE_PRBallHandler',
       'OVERALL_DEF_SCORE_PRRollMan', 'OVERALL_DEF_SCORE_Postup',
       'OVERALL_DEF_SCORE_Spotup', 'OVERALL_DEF_SCORE_Transition','DEF_EFFICIENCY_SCORE_Cut',
       'DEF_EFFICIENCY_SCORE_Handoff', 'DEF_EFFICIENCY_SCORE_Isolation',
       'DEF_EFFICIENCY_SCORE_Misc', 'DEF_EFFICIENCY_SCORE_OffRebound',
       'DEF_EFFICIENCY_SCORE_OffScreen', 'DEF_EFFICIENCY_SCORE_PRBallHandler',
       'DEF_EFFICIENCY_SCORE_PRRollMan', 'DEF_EFFICIENCY_SCORE_Postup',
       'DEF_EFFICIENCY_SCORE_Spotup', 'DEF_EFFICIENCY_SCORE_Transition','DISRUPTION_SCORE_Cut',
       'DISRUPTION_SCORE_Handoff', 'DISRUPTION_SCORE_Isolation',
       'DISRUPTION_SCORE_Misc', 'DISRUPTION_SCORE_OffRebound',
       'DISRUPTION_SCORE_OffScreen', 'DISRUPTION_SCORE_PRBallHandler',
       'DISRUPTION_SCORE_PRRollMan', 'DISRUPTION_SCORE_Postup',
       'DISRUPTION_SCORE_Spotup', 'DISRUPTION_SCORE_Transition',
       'PU_FG2_PCT_60Day', 'PU_FG3_PCT_60Day','PU_FGA_PCT_60Day', 'PU_FGA_rate_60Day',
       'PU_FG3A_rate_60Day', 'CS_FG2_PCT_60Day', 'CS_FG3_PCT_60Day',
       'CS_FGA_rate_60Day', 'CS_FG3A_rate_60Day','PTS_PER_DRIVE', 'DRIVES/MIN_60Day', 'DRIVES_AST_PASS_RATE',
       'DRIVES_PASS_RATE','SHOTS_DRIVE_RATE','FGA_3_AB_60G_Sum', 'FGA_3_LC_60G_Sum',
       'FGA_3_RC_60G_Sum', 'FGA_BC_60G_Sum', 'FGA_Mid_60G_Sum',
       'FGA_Paint_60G_Sum', 'FGA_RA_60G_Sum', 'FGM_3_AB_60G_Sum',
       'FGM_3_LC_60G_Sum', 'FGM_3_RC_60G_Sum', 'FGM_BC_60G_Sum',
       'FGM_Mid_60G_Sum', 'FGM_Paint_60G_Sum', 'FGM_RA_60G_Sum']])

In [837]:
df_ast_cat['FG3A_Corner_Rate'] = df_ast_cat['FG3A']/(df_ast_cat['FGA_3_RC_60G_Sum']+df_ast_cat['FGA_3_LC_60G_Sum'])
df_ast_cat['Corner_FG_PCT'] = (df_ast_cat['FGM_3_RC_60G_Sum']+df_ast_cat['FGM_3_LC_60G_Sum'])/(df_ast_cat['FGA_3_RC_60G_Sum']+df_ast_cat['FGA_3_LC_60G_Sum'])
df_ast_cat['Corner_FGA_rate'] = (df_ast_cat['FGA_3_RC_60G_Sum']+df_ast_cat['FGA_3_LC_60G_Sum'])/df_ast_cat['MIN']

In [838]:
#ast
league_avg_drives_ast = df_ast_cat['DRIVES_AST_PASS_RATE'].mean()
league_avg_drives_pass = df_ast_cat['DRIVES_PASS_RATE'].mean()




league_avg_fg_pct = df_ast_cat['CS_FG3_PCT_60Day'].mean()
league_avg_fga_per_min = df_ast_cat['CS_FG3A_rate_60Day'].mean()
league_avg_screen_fg_pct = df_ast_cat['OVERALL_DEF_SCORE_OffScreen'].mean()
league_avg_pts_per_fgm_pnrman = df_ast_cat['OVERALL_DEF_SCORE_PRRollMan'].mean()
league_avg_pts_per_fgm_cut = df_ast_cat['OVERALL_DEF_SCORE_Cut'].mean()
#league_avg_poss_pct_pnr = df_ast_cat['POSS_PCT_PRRollMan'].mean()
league_avg_fg_pct_PU = df_ast_cat['PU_FG3_PCT_60Day'].mean()
league_avg_fga_per_min_PU = df_ast_cat['PU_FG3A_rate_60Day'].mean()
league_avg_spotup = df_ast_cat['OVERALL_DEF_SCORE_Spotup'].mean()
league_avg_corner_fg_pct = df_ast_cat['Corner_FG_PCT'].mean()
league_avg_corner_fga_per_min = df_ast_cat['Corner_FGA_rate'].mean()

In [839]:


offscreen_score = (
        (df_ast_cat['OVERALL_DEF_SCORE_OffScreen'] / league_avg_screen_fg_pct) # Scale by frequency to reward volume
    )
spotup_score = (
        (df_ast_cat['OVERALL_DEF_SCORE_Spotup'] / league_avg_spotup) # Scale by frequency to reward volume
    )


pnr_score = (
        (df_ast_cat['OVERALL_DEF_SCORE_PRRollMan']/league_avg_pts_per_fgm_pnrman)
    # Scale by frequency to reward volume
    )

cut_score = (
        (df_ast_cat['OVERALL_DEF_SCORE_Cut']/league_avg_pts_per_fgm_cut)
    # Scale by frequency to reward volume
    )

shooting_component = (
        (df_ast_cat['CS_FG3_PCT_60Day'] / league_avg_fg_pct) * 
        (df_ast_cat['CS_FG3A_rate_60Day'] / league_avg_fga_per_min))
shooting_component_pu = (
        (df_ast_cat['PU_FG3_PCT_60Day'] / league_avg_fg_pct_PU) * 
        (df_ast_cat['PU_FG3A_rate_60Day'] / league_avg_fga_per_min_PU))
shooting_component_corner = (
        (df_ast_cat['Corner_FG_PCT'] / league_avg_corner_fg_pct) * 
        (df_ast_cat['Corner_FGA_rate'] / league_avg_corner_fga_per_min))

drive_pass_component = (
        (df_ast_cat['DRIVES_AST_PASS_RATE'] / league_avg_drives_ast) * 
        (df_ast_cat['DRIVES_PASS_RATE'] / league_avg_drives_pass))

In [840]:
df_ast_cat['ast_shooter_score'] = (
        shooting_component * 0.25 + 
        drive_pass_component * 0.25 + # 60% weight on shooting
        offscreen_score * 0.25 +
        spotup_score * 0.25 # 40% weight on movement
    ) * np.log1p(df_ast_cat['MIN']) * 100
average_score = df_ast_cat['ast_shooter_score'].mean()
df_ast_cat['ast_shooter_score'] = (df_ast_cat['ast_shooter_score'] / average_score) * 100

df_ast_cat['ast_bigs_score'] = (
        pnr_score * 0.3 + 
        drive_pass_component * 0.3 + # 60% weight on shooting
        cut_score * 0.3 +
        spotup_score * 0.1 # 40% weight on movement
    ) * np.log1p(df_ast_cat['MIN']) * 100
average_score = df_ast_cat['ast_bigs_score'].mean()
df_ast_cat['ast_bigs_score'] = (df_ast_cat['ast_bigs_score'] / average_score) * 100

In [841]:
df_ast_cat_ = pd.DataFrame(df_ast_cat[['OPPONENT_ID', 'OPPONENT_NAME','ast_shooter_score',
       'ast_bigs_score','DRIVES/MIN_60Day',
       'DRIVES_AST_PASS_RATE', 'DRIVES_PASS_RATE',]])

In [842]:
df_ast_cat_['as_of'] = pd.to_datetime(today)
df_ast_cat_['id'] = df_ast_cat_['as_of'].astype(str)+"_"+df_ast_cat_['OPPONENT_ID'].astype(str)

# Connect to SQLite database
conn = sqlite3.connect('nba_data.db')
cursor = conn.cursor()

# Step 1: Check if the table exists
table_name = "def_ast_scores"
cursor.execute(f"SELECT name FROM sqlite_master WHERE type='table' AND name='{table_name}';")
table_exists = cursor.fetchone()

# Step 2: Create the table if it doesn't exist
if not table_exists:
    df_ast_cat_.to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' created and data inserted.")
else:
    print(f"Table '{table_name}' already exists. Checking for new records...")

    # Step 3: Pull existing IDs from the table
    existing_ids = pd.read_sql(f"SELECT id FROM {table_name}", conn)
    new_data = df_ast_cat_[~df_ast_cat_['id'].isin(existing_ids['id'])]  # Filter for new rows

    # Step 4: Insert new records only
    if not new_data.empty:
        new_data.to_sql(table_name, conn, if_exists='append', index=False)
        print(f"Inserted {len(new_data)} new records into '{table_name}'.")
    else:
        print("No new records to insert.")

# Close the connection
conn.close()

Table 'def_ast_scores' already exists. Checking for new records...
Inserted 30 new records into 'def_ast_scores'.
